# HEIF Encoder Example with C++23 (xeus-cpp)

This notebook demonstrates encoding HEIF files from GeoTIFF sources using libheif with various compression and tiling options.

## Features Demonstrated:
- Non-tiled images (uncompressed and compressed)
- Tiled images using grid/unci/tili schemes
- Pyramid/multi-resolution images
- Various compression methods (HEVC, AV1, uncompressed, DEFLATE, Brotli, etc.)
- GeoTIFF input handling with GDAL
- Detailed HEIF file structure analysis

## Environment:
- Runs in isolated mamba/conda environment
- All libraries installed in $CONDA_PREFIX
- System-independent library loading using dlopen
- Cross-platform path handling

## Cell Re-execution Safe:
All definitions are protected with include guards to prevent redefinition errors.

# SETUP

## Setup: Environment Detection and Global Configuration

In [ ]:
#ifndef HEIF_NOTEBOOK_GLOBALS_H
#define HEIF_NOTEBOOK_GLOBALS_H

#include <iostream>
#include <filesystem>
#include <cstdlib>
#include <string>
#include <vector>

namespace fs = std::filesystem;

// Detect operating system
#if defined(_WIN32) || defined(_WIN64)
    #define HEIF_OS_WINDOWS 1
    #define HEIF_LIB_PREFIX ""
    #define HEIF_LIB_EXT ".dll"
#elif defined(__APPLE__) || defined(__MACH__)
    #define HEIF_OS_MACOS 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".dylib"
#elif defined(__linux__)
    #define HEIF_OS_LINUX 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#else
    #define HEIF_OS_UNKNOWN 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#endif

// Global namespace for persistent environment configuration
namespace heif_notebook {

// Environment configuration structure
struct EnvironmentConfig {
    fs::path conda_prefix;
    fs::path include_path;
    fs::path lib_path;
    std::string os_name;
    std::string lib_extension;
    std::string lib_prefix;
    bool is_conda_env;
    
    EnvironmentConfig() : is_conda_env(false) {
        // Detect OS
#ifdef HEIF_OS_WINDOWS
        os_name = "Windows";
#elif defined(HEIF_OS_MACOS)
        os_name = "macOS";
#elif defined(HEIF_OS_LINUX)
        os_name = "Linux";
#else
        os_name = "Unknown";
#endif
        
        lib_extension = HEIF_LIB_EXT;
        lib_prefix = HEIF_LIB_PREFIX;
        
        // Get CONDA_PREFIX
        const char* conda_prefix_env = std::getenv("CONDA_PREFIX");
        if (conda_prefix_env) {
            conda_prefix = fs::path(conda_prefix_env);
            is_conda_env = true;
            include_path = conda_prefix / "include";
            lib_path = conda_prefix / "lib";
        } else {
            // Fallback to system paths
            is_conda_env = false;
#ifdef HEIF_OS_WINDOWS
            include_path = "C:/Program Files/include";
            lib_path = "C:/Program Files/lib";
#else
            include_path = "/usr/local/include";
            lib_path = "/usr/local/lib";
#endif
        }
    }
    
    void print_info() const {
        std::cout << "\n=== Environment Information ===" << std::endl;
        std::cout << "Operating System: " << os_name << std::endl;
        std::cout << "Library Prefix: " << lib_prefix << std::endl;
        std::cout << "Library Extension: " << lib_extension << std::endl;
        std::cout << "Conda Environment: " << (is_conda_env ? "Yes" : "No") << std::endl;
        
        if (is_conda_env) {
            std::cout << "CONDA_PREFIX: " << conda_prefix << std::endl;
            std::cout << "Include Path: " << include_path << std::endl;
            std::cout << "Library Path: " << lib_path << std::endl;
        }
        std::cout << "\n✓ Environment detected" << std::endl;
    }
};

// Global configuration instance - singleton pattern
inline EnvironmentConfig& get_env_config() {
    static EnvironmentConfig instance;
    return instance;
}

} // namespace heif_notebook

// Initialize and print environment info
heif_notebook::get_env_config().print_info();

#endif // HEIF_NOTEBOOK_GLOBALS_H

## Setup: Dynamic Library Loading with dlopen

In [ ]:
#ifndef HEIF_NOTEBOOK_DLOPEN_H
#define HEIF_NOTEBOOK_DLOPEN_H

#include <iostream>
#include <filesystem>
#include <string>
#include <vector>
#include <map>

// Platform-specific includes for dlopen
#ifdef HEIF_OS_WINDOWS
    #include <windows.h>
    #define RTLD_NOW 0
    #define RTLD_GLOBAL 0
    typedef HMODULE dl_handle_t;
    
    void* dlopen(const char* filename, int flags) {
        return (void*)LoadLibraryA(filename);
    }
    
    const char* dlerror() {
        static char error_msg[256];
        FormatMessageA(FORMAT_MESSAGE_FROM_SYSTEM, NULL, GetLastError(),
                      MAKELANGID(LANG_NEUTRAL, SUBLANG_DEFAULT),
                      error_msg, sizeof(error_msg), NULL);
        return error_msg;
    }
    
    int dlclose(void* handle) {
        return FreeLibrary((HMODULE)handle) ? 0 : -1;
    }
#else
    #include <dlfcn.h>
    typedef void* dl_handle_t;
#endif

namespace heif_notebook {

// Library handle manager
class LibraryLoader {
private:
    std::map<std::string, dl_handle_t> loaded_libraries;
    
public:
    // Construct full library path and find existing library
    std::string get_library_path(const std::string& lib_name) {
        namespace fs = std::filesystem;
        
        // Get environment config
        auto& env = get_env_config();
        
        // Try with different version suffix patterns
        std::vector<std::string> try_patterns = {
            env.lib_prefix + lib_name + env.lib_extension,
        };
        
        // Add versioned patterns for Unix-like systems
#ifndef HEIF_OS_WINDOWS
        // Try .dylib.X or .so.X patterns
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + std::to_string(ver));
        }
        // Try .X.Y.Z patterns
        for (int major = 1; major <= 5; major++) {
            for (int minor = 0; minor <= 9; minor++) {
                try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + 
                                      std::to_string(major) + "." + std::to_string(minor));
            }
        }
        // Try .X.dylib patterns (alternate macOS versioning)
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + "." + std::to_string(ver) + env.lib_extension);
        }
#endif
        
        // Search for existing library file
        for (const auto& pattern : try_patterns) {
            fs::path lib_path = env.lib_path / pattern;
            if (fs::exists(lib_path)) {
                return lib_path.string();
            }
        }
        
        // Return default path (for error reporting)
        fs::path default_path = env.lib_path / (env.lib_prefix + lib_name + env.lib_extension);
        return default_path.string();
    }
    
    // Check if library exists
    bool library_exists(const std::string& lib_name) {
        namespace fs = std::filesystem;
        std::string lib_path = get_library_path(lib_name);
        return fs::exists(lib_path);
    }
    
    // Load a library
    bool load_library(const std::string& lib_name, bool required = true) {
        // Check if already loaded
        if (loaded_libraries.find(lib_name) != loaded_libraries.end()) {
            std::cout << "  ℹ " << lib_name << " already loaded" << std::endl;
            return true;
        }
        
        std::cout << "  Loading: " << lib_name << std::endl;
        
        std::string lib_path = get_library_path(lib_name);
        std::cout << "    Path: " << lib_path << std::endl;
        
        namespace fs = std::filesystem;
        if (!fs::exists(lib_path)) {
            if (required) {
                std::cout << "    ✗ Not found (Required)" << std::endl;
            } else {
                std::cout << "    ⚠ Not found (Optional)" << std::endl;
            }
            return false;
        }
        
        dl_handle_t handle = (dl_handle_t)dlopen(lib_path.c_str(), RTLD_NOW | RTLD_GLOBAL);
        
        if (!handle) {
            const char* error = dlerror();
            std::cout << "    ✗ Failed to load: " << (error ? error : "Unknown error") << std::endl;
            return false;
        }
        
        loaded_libraries[lib_name] = handle;
        std::cout << "    ✓ Loaded successfully" << std::endl;
        return true;
    }
    
    // Load multiple libraries in order
    void load_libraries(const std::vector<std::pair<std::string, bool>>& libraries) {
        std::cout << "\n=== Loading Libraries ===" << std::endl;
        for (const auto& [lib_name, required] : libraries) {
            load_library(lib_name, required);
        }
        std::cout << "\n✓ Library loading complete" << std::endl;
    }
    
    // Get count of loaded libraries
    size_t loaded_count() const {
        return loaded_libraries.size();
    }
    
    // Unload all libraries
    ~LibraryLoader() {
        for (auto& [name, handle] : loaded_libraries) {
            if (handle) {
                dlclose(handle);
            }
        }
    }
};

// Global library loader instance - singleton pattern
inline LibraryLoader& get_library_loader() {
    static LibraryLoader instance;
    return instance;
}

} // namespace heif_notebook

std::cout << "\n✓ Library loader initialized" << std::endl;

#endif // HEIF_NOTEBOOK_DLOPEN_H

## Load All Required Libraries

In [ ]:
#ifndef HEIF_NOTEBOOK_LOAD_ALL_H
#define HEIF_NOTEBOOK_LOAD_ALL_H

// Define library loading order (dependencies first)
std::vector<std::pair<std::string, bool>> libraries_to_load = {
    // Core compression libraries (no dependencies)
    {"z", true},              // zlib - required by many libraries
    {"bz2", false},           // bzip2 - optional
    {"lzma", false},          // LZMA - optional
    {"zstd", false},          // Zstandard - optional
    {"brotlicommon", false},  // Brotli common - optional
    {"brotlidec", false},     // Brotli decoder - optional
    {"brotlienc", false},     // Brotli encoder - optional
    
    // Image format libraries
    {"jpeg", false},          // JPEG - used by TIFF/GDAL
    {"png", false},           // PNG - used by GDAL
    {"png16", false},         // PNG 1.6 - alternative name
    {"webp", false},          // WebP - optional
    
    // TIFF library (depends on zlib, jpeg, etc.)
    {"tiff", true},           // libtiff - required
    
    // NOTE: We skip loading libgeotiff separately since GDAL includes GeoTIFF support
    // Loading both causes header conflicts
    
    // Geospatial libraries
    {"proj", false},          // PROJ - used by GDAL
    {"sqlite3", false},       // SQLite - used by GDAL
    {"curl", false},          // cURL - used by GDAL
    {"expat", false},         // Expat XML parser - used by GDAL
    {"xml2", false},          // libxml2 - used by GDAL
    
    // GDAL (includes GeoTIFF support, depends on many of the above)
    {"gdal", true},           // GDAL - required
    
    // Video codec libraries (for HEIF)
    {"x265", false},          // x265 HEVC encoder - optional
    {"de265", false},         // de265 HEVC decoder - optional
    {"aom", false},           // AOM AV1 codec - optional
    {"dav1d", false},         // dav1d AV1 decoder - optional
    {"rav1e", false},         // rav1e AV1 encoder - optional
    {"SvtAv1Enc", false},     // SVT-AV1 encoder - optional
    {"SvtAv1Dec", false},     // SVT-AV1 decoder - optional
    
    // HEIF library (depends on codecs)
    {"heif", true},           // libheif - required
};

// Load all libraries
heif_notebook::get_library_loader().load_libraries(libraries_to_load);

// Report summary
std::cout << "\nSuccessfully loaded " << heif_notebook::get_library_loader().loaded_count() 
          << " libraries" << std::endl;

#endif // HEIF_NOTEBOOK_LOAD_ALL_H

## Standard C++23 Libraries

In [ ]:
#ifndef HEIF_NOTEBOOK_STD_HEADERS_H
#define HEIF_NOTEBOOK_STD_HEADERS_H

// C++23 Standard Library
#include <iostream>
#include <fstream>
#include <sstream>
#include <iomanip>
#include <format>      // C++20/23 formatting
#include <print>       // C++23 print functions
#include <expected>    // C++23 error handling
#include <optional>
#include <variant>
#include <memory>
#include <vector>
#include <array>
#include <string>
#include <string_view>
#include <span>        // C++20 span
#include <map>
#include <ranges>      // C++20/23 ranges
#include <algorithm>
#include <functional>
#include <concepts>    // C++20 concepts
#include <cstring>
#include <cstdint>
#include <stdexcept>
#include <source_location> // C++20 source location
#include <filesystem>  // C++17 filesystem

// Check C++ version
static_assert(__cplusplus >= 202002L, "C++20 or later required");

std::cout << "\nC++ Version: " << __cplusplus << std::endl;
std::cout << "✓ Standard C++ libraries loaded" << std::endl;

#endif // HEIF_NOTEBOOK_STD_HEADERS_H

## Load Library Headers - Single Combined Cell



**IMPORTANT**: All library headers must be loaded in a single cell to avoid redefinition conflicts.

**NOTE**: We do NOT load libgeotiff headers separately. GDAL includes GeoTIFF support internally,
and loading both libgeotiff and GDAL headers causes `CE_None` enum redefinition conflicts.

In [ ]:
#ifndef HEIF_NOTEBOOK_ALL_HEADERS_H
#define HEIF_NOTEBOOK_ALL_HEADERS_H

// ============================================================================
// Load all library headers in correct order (dependencies first)
// NOTE: Do NOT include libgeotiff headers - GDAL includes GeoTIFF support
// ============================================================================

// 1. zlib (no dependencies)
#ifndef HEIF_ZLIB_LOADED
#define HEIF_ZLIB_LOADED
#include <zlib.h>
#endif

// 2. libtiff (depends on zlib)
#ifndef HEIF_TIFF_LOADED
#define HEIF_TIFF_LOADED
#include <tiffio.h>
#include <tiffio.hxx>
#endif

// 3. GDAL (depends on TIFF, includes GeoTIFF support internally)
#ifndef HEIF_GDAL_LOADED
#define HEIF_GDAL_LOADED
#ifdef __has_include
#if __has_include(<gdal.h>)
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#elif __has_include(<gdal/gdal.h>)
    #include <gdal/gdal.h>
    #include <gdal/gdal_priv.h>
    #include <gdal/cpl_conv.h>
    #include <gdal/cpl_string.h>
    #include <gdal/ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX "gdal/"
#else
    #error "GDAL headers not found. Check installation."
#endif
#else
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#endif
#endif // HEIF_GDAL_LOADED

// 4. libheif (may depend on codecs)
#ifndef HEIF_LIBHEIF_LOADED
#define HEIF_LIBHEIF_LOADED

// Enable experimental features
#ifndef HEIF_ENABLE_EXPERIMENTAL_FEATURES
#define HEIF_ENABLE_EXPERIMENTAL_FEATURES 1
#endif

#ifndef WITH_UNCOMPRESSED_CODEC
#define WITH_UNCOMPRESSED_CODEC 1
#endif

#ifdef __has_include
#if __has_include(<libheif/heif.h>)
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#elif __has_include(<heif.h>)
    #include <heif.h>
    #include <heif_properties.h>
    #include <heif_items.h>
    #include <heif_experimental.h>
    #include <heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX ""
#else
    #error "libheif headers not found. Check installation."
#endif
#else
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#endif
#endif // HEIF_LIBHEIF_LOADED

// 5. Optional compression libraries
#ifndef HEIF_OPTIONAL_LIBS_LOADED
#define HEIF_OPTIONAL_LIBS_LOADED

#ifdef __has_include
#if __has_include(<brotli/encode.h>)
    #ifndef HEIF_HAVE_BROTLI
    #define HEIF_HAVE_BROTLI 1
    #include <brotli/encode.h>
    #include <brotli/decode.h>
    #endif
#endif

#if __has_include(<zstd.h>)
    #ifndef HEIF_HAVE_ZSTD
    #define HEIF_HAVE_ZSTD 1
    #include <zstd.h>
    #endif
#endif
#endif

#endif // HEIF_OPTIONAL_LIBS_LOADED

// ============================================================================
// Check and report loaded headers
// ============================================================================

std::cout << "\n=== Library Headers Loaded ===" << std::endl;
std::cout << "✓ zlib" << std::endl;
std::cout << "✓ libtiff" << std::endl;
std::cout << "✓ GDAL (include prefix: " << HEIF_GDAL_INCLUDE_PREFIX << ")" << std::endl;
std::cout << "  (GeoTIFF support included via GDAL)" << std::endl;
std::cout << "✓ libheif (include prefix: " << HEIF_INCLUDE_PREFIX << ")" << std::endl;

#ifdef HEIF_HAVE_BROTLI
std::cout << "✓ Brotli" << std::endl;
#else
std::cout << "⚠ Brotli (not available)" << std::endl;
#endif

#ifdef HEIF_HAVE_ZSTD
std::cout << "✓ ZSTD" << std::endl;
#else
std::cout << "⚠ ZSTD (not available)" << std::endl;
#endif

std::cout << "\n✓ All library headers loaded successfully" << std::endl;

#endif // HEIF_NOTEBOOK_ALL_HEADERS_H

### GDAL Headers

In [ ]:
#ifndef HEIF_NOTEBOOK_GDAL_H
#define HEIF_NOTEBOOK_GDAL_H

// Try different include paths for GDAL
#ifdef __has_include
#if __has_include(<gdal.h>)
    // Conda/system installation without prefix
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX ""
#elif __has_include(<gdal/gdal.h>)
    // System installation with prefix
    #include <gdal/gdal.h>
    #include <gdal/gdal_priv.h>
    #include <gdal/cpl_conv.h>
    #include <gdal/cpl_string.h>
    #include <gdal/ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX "gdal/"
#else
    #error "GDAL headers not found. Check installation."
#endif
#else
    // Fallback: try without prefix
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX ""
#endif

void check_gdal_capabilities() {
    using namespace std;
    
    println("\n=== GDAL Information ===");
    println("Include prefix: {}", GDAL_INCLUDE_PREFIX);
    println("Version: {}", GDALVersionInfo("RELEASE_NAME"));
    println("Version Number: {}", GDALVersionInfo("VERSION_NUM"));
    
    GDALAllRegister();
    
    int driver_count = GDALGetDriverCount();
    println("\nRegistered Drivers: {}", driver_count);
    
    println("\nImportant Raster Format Support:");
    
    constexpr std::array<std::string_view, 11> important_formats = {
        "GTiff", "COG", "JPEG", "PNG", "HFA", "netCDF",
        "JP2OpenJPEG", "WEBP", "ZMAP", "NITF", "HDF5"
    };
    
    for (const auto& format : important_formats) {
        GDALDriverH driver = GDALGetDriverByName(format.data());
        if (driver) {
            const char* long_name = GDALGetMetadataItem(driver, "DMD_LONGNAME", nullptr);
            println("  ✓ {:15s} - {}", format, long_name ? long_name : "");
        }
    }
    
    println("\nGeoTIFF Compression Methods:");
    GDALDriverH gtiff_driver = GDALGetDriverByName("GTiff");
    if (gtiff_driver) {
        const char* compress_list = GDALGetMetadataItem(gtiff_driver, "DMD_CREATIONOPTIONLIST", nullptr);
        if (compress_list) {
            std::string_view compress_str(compress_list);
            
            constexpr std::array<std::string_view, 10> compression_methods = {
                "NONE", "JPEG", "LZW", "PACKBITS", "DEFLATE",
                "LZMA", "ZSTD", "LERC", "WEBP", "JXL"
            };
            
            for (const auto& method : compression_methods) {
                auto search_str = format("<Value>{}</Value>", method);
                if (compress_str.find(search_str) != std::string_view::npos) {
                    println("  ✓ {}", method);
                }
            }
        }
    }
}

check_gdal_capabilities();

std::println("✓ GDAL headers loaded");

#endif // HEIF_NOTEBOOK_GDAL_H

### Verify Library Capabilities

In [ ]:
#ifndef HEIF_NOTEBOOK_VERIFY_H
#define HEIF_NOTEBOOK_VERIFY_H

void verify_all_libraries() {
    using namespace std;
    
    // libheif
    println("\n=== libheif ===");
    println("Version: {}", heif_get_version());
    
    // GDAL
    println("\n=== GDAL ===");
    println("Version: {}", GDALVersionInfo("RELEASE_NAME"));
    GDALAllRegister();
    println("Registered Drivers: {}", GDALGetDriverCount());
    
    // Check for GeoTIFF driver in GDAL
    GDALDriverH gtiff_driver = GDALGetDriverByName("GTiff");
    if (gtiff_driver) {
        println("✓ GeoTIFF support available via GDAL");
    } else {
        println("⚠ GeoTIFF driver not found in GDAL");
    }
    
    // TIFF
    println("\n=== libtiff ===");
    const char* tiff_version = TIFFGetVersion();
    if (tiff_version) {
        string_view version_str(tiff_version);
        auto pos = version_str.find("Version");
        if (pos != string_view::npos) {
            auto start = version_str.find_first_of("0123456789", pos);
            auto end = version_str.find("\n", start);
            if (start != string_view::npos) {
                println("Version: {}", version_str.substr(start, end - start));
            }
        }
    }
    
    // zlib
    println("\n=== zlib ===");
    println("Version: {}", zlibVersion());
    
#ifdef HEIF_HAVE_BROTLI
    println("\n=== Brotli ===");
    uint32_t brotli_ver = BrotliEncoderVersion();
    println("Version: {}.{}.{}", brotli_ver >> 24, (brotli_ver >> 12) & 0xFFF, brotli_ver & 0xFFF);
#endif
    
#ifdef HEIF_HAVE_ZSTD
    println("\n=== ZSTD ===");
    println("Version: {}", ZSTD_versionString());
#endif
    
    println("\n✓ All libraries verified");
}

verify_all_libraries();

#endif // HEIF_NOTEBOOK_VERIFY_H

## Important Notes:

### Why No Separate libgeotiff?

1. **GDAL includes GeoTIFF**: Modern GDAL includes comprehensive GeoTIFF support via the GTiff driver
2. **Header conflicts**: libgeotiff and GDAL both define CPL error enums (CE_None, etc.)
3. **Redundant**: Loading both provides no additional functionality

### GeoTIFF Support via GDAL:

GDAL provides full GeoTIFF support including:
- Reading/writing GeoTIFF tags
- Coordinate system handling
- Projection information
- All standard GeoTIFF metadata

### Header Loading Order:

1. **zlib** - Foundation compression library
2. **libtiff** - TIFF file I/O
3. **GDAL** - Includes GeoTIFF support, depends on TIFF
4. **libheif** - HEIF container format
5. **Optional libs** - Brotli, ZSTD, etc.

## Compression Algorithm Enumerations with C++23 Features

In [ ]:
#ifndef HEIF_COMPRESSION_TYPES_H
#define HEIF_COMPRESSION_TYPES_H

// C++23: Use enum class with underlying type specification
enum class ExtendedCompressionType : uint8_t {
    None = 0,
    HEVC,
    AV1,
    VVC,
    AVC,
    JPEG,
    JPEG2000,
    HTJ2K,
    Uncompressed,
    // Additional GDAL-compatible compressions
    DEFLATE,
    ZLIB,
    Brotli,
    ZSTD,
    LZW,
    WebP,
    LERC,
    PackBits
};

// C++23: Use designated initializers and constexpr
struct CompressionInfo {
    ExtendedCompressionType type;
    std::string_view name;
    std::string_view description;
    bool lossless;
    bool supported_by_libheif;
    bool supported_by_gdal;
};

// C++23: constexpr vector-like container
constexpr inline std::array<CompressionInfo, 16> COMPRESSION_ALGORITHMS = {{
    {ExtendedCompressionType::None, "NONE", "No compression", true, true, true},
    {ExtendedCompressionType::HEVC, "HEVC", "H.265/HEVC video codec", false, true, false},
    {ExtendedCompressionType::AV1, "AV1", "AV1 video codec", false, true, false},
    {ExtendedCompressionType::VVC, "VVC", "H.266/VVC video codec", false, true, false},
    {ExtendedCompressionType::AVC, "AVC", "H.264/AVC video codec", false, true, false},
    {ExtendedCompressionType::JPEG, "JPEG", "JPEG compression", false, true, true},
    {ExtendedCompressionType::JPEG2000, "JPEG2000", "JPEG 2000", false, true, true},
    {ExtendedCompressionType::HTJ2K, "HTJ2K", "High Throughput JPEG 2000", false, true, false},
    {ExtendedCompressionType::DEFLATE, "DEFLATE", "DEFLATE (zlib) compression", true, true, true},
    {ExtendedCompressionType::ZLIB, "ZLIB", "ZLIB compression", true, true, true},
    {ExtendedCompressionType::Brotli, "BROTLI", "Brotli compression", true, true, false},
    {ExtendedCompressionType::ZSTD, "ZSTD", "Zstandard compression", true, false, true},
    {ExtendedCompressionType::LZW, "LZW", "LZW compression", true, false, true},
    {ExtendedCompressionType::WebP, "WEBP", "WebP compression", false, false, true},
    {ExtendedCompressionType::LERC, "LERC", "Limited Error Raster Compression", true, false, true},
    {ExtendedCompressionType::PackBits, "PACKBITS", "PackBits compression", true, false, true}
}};

void list_all_compression_algorithms() {
    using namespace std;
    
    println("\n=== Available Compression Algorithms ===");
    println("{:15s} {:40s} {:10s} {:10s} {:10s}",
            "Name", "Description", "Lossless", "libheif", "GDAL");
    println("{}", string(85, '-'));
    
    // C++23: Use ranges with views
    for (const auto& comp : COMPRESSION_ALGORITHMS) {
        println("{:15s} {:40s} {:10s} {:10s} {:10s}",
                comp.name,
                comp.description,
                comp.lossless ? "Yes" : "No",
                comp.supported_by_libheif ? "Yes" : "No",
                comp.supported_by_gdal ? "Yes" : "No");
    }
}

std::println("Compression types defined.");

#endif // HEIF_COMPRESSION_TYPES_H

## Helper Functions with C++23 Error Handling (Updated)

In [ ]:
#ifndef HEIF_HELPER_FUNCTIONS_H
#define HEIF_HELPER_FUNCTIONS_H

// C++23: Use std::expected for error handling
using HeifResult = std::expected<void, std::string>;

template<typename T>
using HeifExpected = std::expected<T, std::string>;

// Helper function to check errors with source location
HeifResult check_error(heif_error error, 
                       std::string_view context,
                       std::source_location loc = std::source_location::current()) {
    if (error.code != heif_error_Ok) {
        auto err_msg = std::format("Error in {} at {}:{}: {}",
                                   context,
                                   loc.file_name(),
                                   loc.line(),
                                   error.message);
        return std::unexpected(err_msg);
    }
    return {};
}

// Simpler version without std::expected for immediate error reporting
void check_error_simple(heif_error error, std::string_view context) {
    if (error.code != heif_error_Ok) {
        std::cerr << "Error in " << context << ": " << error.message << std::endl;
        throw std::runtime_error(error.message);
    }
}

// Helper to list available encoders with modern formatting
void list_encoders(heif_compression_format format) {
    using namespace std;
    
    const heif_encoder_descriptor* descriptors[10];
    int count = heif_get_encoder_descriptors(format, nullptr, descriptors, 10);
    
    println("Available encoders for format {}:", static_cast<int>(format));
    
    // C++23: Use ranges to transform and print
    for (int i : std::views::iota(0, count)) {
        println("  - {}: {}",
                heif_encoder_descriptor_get_id_name(descriptors[i]),
                heif_encoder_descriptor_get_name(descriptors[i]));
    }
}

// C++23: Use constexpr mapping function
constexpr heif_compression_format to_heif_compression_format(ExtendedCompressionType type) {
    using enum ExtendedCompressionType;
    
    switch(type) {
        case HEVC: return heif_compression_HEVC;
        case AV1: return heif_compression_AV1;
        case VVC: return heif_compression_VVC;
        case AVC: return heif_compression_AVC;
        case JPEG: return heif_compression_JPEG;
        case JPEG2000: return heif_compression_JPEG2000;
        case HTJ2K: return heif_compression_HTJ2K;
        case Uncompressed:
        case DEFLATE:
        case ZLIB:
        case Brotli:
            return heif_compression_uncompressed;
        default:
            return heif_compression_undefined;
    }
}

// C++23: Use constexpr for compile-time evaluation
constexpr heif_unci_compression to_unci_compression(ExtendedCompressionType type) {
    using enum ExtendedCompressionType;
    
    switch(type) {
        case DEFLATE: return heif_unci_compression_deflate;
        case ZLIB: return heif_unci_compression_zlib;
        case Brotli: return heif_unci_compression_brotli;
        default: return heif_unci_compression_off;
    }
}

// Get compression info by type - IMPORTANT: Make this inline to avoid multiple definition errors
inline const CompressionInfo* get_compression_info(ExtendedCompressionType type) {
    for (const auto& info : COMPRESSION_ALGORITHMS) {
        if (info.type == type) {
            return &info;
        }
    }
    return nullptr;
}

// Get compression name as string
inline std::string_view get_compression_name(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    return info ? info->name : "Unknown";
}

// Check if compression is lossless
inline bool is_lossless_compression(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    return info ? info->lossless : false;
}

std::cout << "Helper functions defined." << std::endl;

#endif // HEIF_HELPER_FUNCTIONS_H

## GDAL-based GeoTIFF Loading

In [ ]:
#ifndef HEIF_GDAL_LOADER_H
#define HEIF_GDAL_LOADER_H

struct GDALImageData {
    int width;
    int height;
    int bands;
    GDALDataType data_type;
    std::vector<std::vector<uint8_t>> band_data;
    std::string projection;
    double geo_transform[6];
    std::map<std::string, std::string> metadata;
};

class GDALImageLoader {
private:
    static bool gdal_initialized;
    
public:
    static void initialize() {
        if (!gdal_initialized) {
            GDALAllRegister();
            gdal_initialized = true;
            std::cout << "GDAL initialized with " << GDALGetDriverCount() << " drivers." << std::endl;
        }
    }
    
    static GDALImageData load(const char* filename) {
        initialize();
        
        GDALDataset* dataset = (GDALDataset*)GDALOpen(filename, GA_ReadOnly);
        if (!dataset) {
            throw std::runtime_error("Cannot open file with GDAL: " + std::string(filename));
        }
        
        GDALImageData data;
        data.width = dataset->GetRasterXSize();
        data.height = dataset->GetRasterYSize();
        data.bands = dataset->GetRasterCount();
        
        std::cout << "Loaded: " << data.width << "x" << data.height 
                  << ", " << data.bands << " bands" << std::endl;
        
        // Get projection and geotransform
        const char* proj = dataset->GetProjectionRef();
        if (proj) {
            data.projection = proj;
        }
        
        dataset->GetGeoTransform(data.geo_transform);
        
        // Get metadata
        char** metadata_list = dataset->GetMetadata();
        if (metadata_list) {
            for (int i = 0; metadata_list[i] != nullptr; i++) {
                std::string item(metadata_list[i]);
                size_t eq_pos = item.find('=');
                if (eq_pos != std::string::npos) {
                    std::string key = item.substr(0, eq_pos);
                    std::string value = item.substr(eq_pos + 1);
                    data.metadata[key] = value;
                }
            }
        }
        
        // Read band data
        for (int band_idx = 1; band_idx <= data.bands; band_idx++) {
            GDALRasterBand* band = dataset->GetRasterBand(band_idx);
            data.data_type = band->GetRasterDataType();
            
            int pixel_size = GDALGetDataTypeSizeBytes(data.data_type);
            std::vector<uint8_t> band_buffer(data.width * data.height * pixel_size);
            
            CPLErr err = band->RasterIO(GF_Read, 0, 0, data.width, data.height,
                                       band_buffer.data(), data.width, data.height,
                                       data.data_type, 0, 0);
            
            if (err != CE_None) {
                GDALClose(dataset);
                throw std::runtime_error("Failed to read band " + std::to_string(band_idx));
            }
            
            data.band_data.push_back(std::move(band_buffer));
        }
        
        GDALClose(dataset);
        return data;
    }
    
    static heif_image* to_heif_image(const GDALImageData& data, int output_bit_depth = 8) {
        heif_image* image = nullptr;
        heif_colorspace colorspace = (data.bands >= 3) ? heif_colorspace_RGB : heif_colorspace_monochrome;
        heif_chroma chroma = (data.bands >= 3) ? heif_chroma_444 : heif_chroma_monochrome;
        
        heif_error error = heif_image_create(data.width, data.height, colorspace, chroma, &image);
        check_error(error, "heif_image_create");
        
        // Add planes and copy data
        if (data.bands >= 3) {
            heif_channel channels[] = {heif_channel_R, heif_channel_G, heif_channel_B};
            for (int i = 0; i < 3; i++) {
                heif_image_add_plane(image, channels[i], data.width, data.height, output_bit_depth);
                
                int stride;
                uint8_t* plane_data = heif_image_get_plane(image, channels[i], &stride);
                
                // Simple copy for 8-bit data
                if (data.data_type == GDT_Byte) {
                    for (int y = 0; y < data.height; y++) {
                        memcpy(plane_data + y * stride,
                               data.band_data[i].data() + y * data.width,
                               data.width);
                    }
                }
            }
            
            // Add alpha if 4 bands
            if (data.bands >= 4) {
                heif_image_add_plane(image, heif_channel_Alpha, data.width, data.height, output_bit_depth);
                int stride;
                uint8_t* alpha_data = heif_image_get_plane(image, heif_channel_Alpha, &stride);
                
                if (data.data_type == GDT_Byte) {
                    for (int y = 0; y < data.height; y++) {
                        memcpy(alpha_data + y * stride,
                               data.band_data[3].data() + y * data.width,
                               data.width);
                    }
                }
            }
        } else {
            heif_image_add_plane(image, heif_channel_Y, data.width, data.height, output_bit_depth);
            
            int stride;
            uint8_t* plane_data = heif_image_get_plane(image, heif_channel_Y, &stride);
            
            if (data.data_type == GDT_Byte) {
                for (int y = 0; y < data.height; y++) {
                    memcpy(plane_data + y * stride,
                           data.band_data[0].data() + y * data.width,
                           data.width);
                }
            }
        }
        
        return image;
    }
};

bool GDALImageLoader::gdal_initialized = false;

std::cout << "GDAL image loader defined." << std::endl;

#endif // HEIF_GDAL_LOADER_H

## HEIF File Structure Analyzer

In [ ]:
#ifndef HEIF_FILE_ANALYZER_H
#define HEIF_FILE_ANALYZER_H

class HEIFFileAnalyzer {
private:
    struct BoxInfo {
        uint64_t offset;
        uint64_t size;
        std::string type;
        std::string description;
        int level;
        std::vector<BoxInfo> children;
    };
    
    static std::string fourcc_to_string(uint32_t fourcc) {
        char str[5];
        str[0] = (fourcc >> 24) & 0xFF;
        str[1] = (fourcc >> 16) & 0xFF;
        str[2] = (fourcc >> 8) & 0xFF;
        str[3] = fourcc & 0xFF;
        str[4] = '\0';
        return std::string(str);
    }
    
    static uint32_t read_uint32(std::ifstream& file) {
        uint32_t value;
        file.read(reinterpret_cast<char*>(&value), 4);
        // Convert from big-endian
        return ((value & 0xFF) << 24) | ((value & 0xFF00) << 8) |
               ((value & 0xFF0000) >> 8) | ((value & 0xFF000000) >> 24);
    }
    
    static uint64_t read_uint64(std::ifstream& file) {
        uint64_t value;
        file.read(reinterpret_cast<char*>(&value), 8);
        // Convert from big-endian
        return ((value & 0xFF) << 56) | ((value & 0xFF00) << 40) |
               ((value & 0xFF0000) << 24) | ((value & 0xFF000000) << 8) |
               ((value & 0xFF00000000) >> 8) | ((value & 0xFF0000000000) >> 24) |
               ((value & 0xFF000000000000) >> 40) | ((value & 0xFF00000000000000) >> 56);
    }
    
    static std::string get_box_description(const std::string& type) {
        static const std::map<std::string, std::string> descriptions = {
            {"ftyp", "File Type Box - Brand identification"},
            {"meta", "Meta Box - Container for metadata"},
            {"hdlr", "Handler Box - Media handler type"},
            {"pitm", "Primary Item Box - Primary item reference"},
            {"iloc", "Item Location Box - Item data locations"},
            {"iinf", "Item Information Box - Item information entries"},
            {"iprp", "Item Properties Box - Item properties container"},
            {"ipco", "Item Property Container Box - Property definitions"},
            {"ipma", "Item Property Association Box - Property associations"},
            {"iref", "Item Reference Box - Item relationships"},
            {"idat", "Item Data Box - Embedded item data"},
            {"mdat", "Media Data Box - Media/image data"},
            {"grpl", "Group List Box - Entity groups"},
            {"dinf", "Data Information Box - Data reference info"},
            {"dref", "Data Reference Box - Data entry references"},
            {"ispe", "Image Spatial Extents Property - Image dimensions"},
            {"colr", "Color Information Box - Color profile"},
            {"pixi", "Pixel Information Property - Bits per channel"},
            {"auxC", "Auxiliary Type Property - Auxiliary image type"},
            {"hvcC", "HEVC Configuration - HEVC decoder config"},
            {"av1C", "AV1 Configuration - AV1 decoder config"},
            {"vvcC", "VVC Configuration - VVC decoder config"},
            {"avcC", "AVC Configuration - AVC decoder config"},
            {"tols", "Tiled Output Layer Set - Tiling info"},
            {"unci", "Uncompressed Image - ISO 23001-17"},
            {"tili", "Tiled Image - Tiling structure"},
            {"grid", "Image Grid - Grid layout"},
            {"iovl", "Image Overlay - Overlay composition"},
            {"Exif", "Exif Metadata - Camera/image metadata"},
            {"mime", "MIME Type Data - MIME content"},
            {"clli", "Content Light Level Info - HDR light levels"},
            {"pasp", "Pixel Aspect Ratio - Aspect ratio info"}
        };
        
        auto it = descriptions.find(type);
        if (it != descriptions.end()) {
            return it->second;
        }
        return "Unknown box type";
    }
    
    static BoxInfo read_box(std::ifstream& file, uint64_t max_offset, int level = 0) {
        BoxInfo box;
        box.level = level;
        box.offset = file.tellg();
        
        if (box.offset >= max_offset) {
            box.size = 0;
            return box;
        }
        
        uint32_t size32 = read_uint32(file);
        uint32_t type = read_uint32(file);
        box.type = fourcc_to_string(type);
        
        if (size32 == 1) {
            box.size = read_uint64(file);
        } else if (size32 == 0) {
            box.size = max_offset - box.offset;
        } else {
            box.size = size32;
        }
        
        box.description = get_box_description(box.type);
        
        // Check if this is a container box
        static const std::vector<std::string> container_boxes = {
            "meta", "iprp", "ipco", "iinf", "dinf", "grpl"
        };
        
        bool is_container = std::find(container_boxes.begin(), 
                                     container_boxes.end(), 
                                     box.type) != container_boxes.end();
        
        if (is_container) {
            uint64_t content_start = file.tellg();
            uint64_t content_end = box.offset + box.size;
            
            // Skip version/flags for full boxes
            if (box.type == "meta") {
                file.seekg(4, std::ios::cur);
                content_start = file.tellg();
            }
            
            while (file.tellg() < content_end && file.good()) {
                BoxInfo child = read_box(file, content_end, level + 1);
                if (child.size == 0) break;
                box.children.push_back(child);
                file.seekg(child.offset + child.size);
            }
        }
        
        return box;
    }
    
    static void print_box(const BoxInfo& box, uint64_t total_size) {
        std::string indent(box.level * 2, ' ');
        double percentage = (box.size * 100.0) / total_size;
        
        std::cout << indent
                  << "[" << box.type << "] "
                  << "Offset: 0x" << std::hex << box.offset << std::dec
                  << ", Size: " << box.size << " bytes"
                  << " (" << std::fixed << std::setprecision(2) << percentage << "%)";
        
        if (!box.description.empty()) {
            std::cout << "\n" << indent << "  ↳ " << box.description;
        }
        std::cout << std::endl;
        
        for (const auto& child : box.children) {
            print_box(child, total_size);
        }
    }
    
public:
    static void analyze(const char* filename) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "HEIF File Structure Analysis: " << filename << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
        std::ifstream file(filename, std::ios::binary);
        if (!file) {
            std::cerr << "Cannot open file: " << filename << std::endl;
            return;
        }
        
        // Get file size
        file.seekg(0, std::ios::end);
        uint64_t file_size = file.tellg();
        file.seekg(0, std::ios::beg);
        
        std::cout << "File size: " << file_size << " bytes (" 
                  << (file_size / 1024.0) << " KB)\n" << std::endl;
        
        std::vector<BoxInfo> boxes;
        
        while (file.tellg() < file_size && file.good()) {
            BoxInfo box = read_box(file, file_size, 0);
            if (box.size == 0) break;
            boxes.push_back(box);
            file.seekg(box.offset + box.size);
        }
        
        std::cout << "Box Structure:\n" << std::endl;
        for (const auto& box : boxes) {
            print_box(box, file_size);
        }
        
        // Summary statistics
        std::cout << "\n" << std::string(80, '-') << std::endl;
        std::cout << "Summary Statistics\n" << std::endl;
        
        std::map<std::string, uint64_t> box_sizes;
        std::function<void(const BoxInfo&)> collect_sizes = [&](const BoxInfo& box) {
            box_sizes[box.type] += box.size;
            for (const auto& child : box.children) {
                collect_sizes(child);
            }
        };
        
        for (const auto& box : boxes) {
            collect_sizes(box);
        }
        
        std::vector<std::pair<std::string, uint64_t>> sorted_sizes(box_sizes.begin(), box_sizes.end());
        std::sort(sorted_sizes.begin(), sorted_sizes.end(),
                 [](const auto& a, const auto& b) { return a.second > b.second; });
        
        std::cout << std::left << std::setw(20) << "Box Type" 
                  << std::setw(15) << "Total Size"
                  << std::setw(15) << "Percentage" << std::endl;
        std::cout << std::string(50, '-') << std::endl;
        
        for (const auto& [type, size] : sorted_sizes) {
            double percentage = (size * 100.0) / file_size;
            std::cout << std::left << std::setw(20) << type
                      << std::setw(15) << size
                      << std::fixed << std::setprecision(2) << percentage << "%"
                      << std::endl;
        }
        
        std::cout << "\n" << std::string(80, '=') << "\n" << std::endl;
    }
};

std::cout << "HEIF file analyzer defined." << std::endl;

#endif // HEIF_FILE_ANALYZER_H

## Encoding Functions: Non-Tiled Images

In [ ]:
#ifndef HEIF_ENCODE_NONTILED_H
#define HEIF_ENCODE_NONTILED_H

void encode_nontiled(const char* input_file, const char* output_file,
                    ExtendedCompressionType compression,
                    int quality = 50,
                    int output_bit_depth = 8) {
    
    const CompressionInfo* comp_info = nullptr;
    for (const auto& info : COMPRESSION_ALGORITHMS) {
        if (info.type == compression) {
            comp_info = &info;
            break;
        }
    }
    
    std::cout << "\n=== Encoding Non-Tiled Image ===" << std::endl;
    std::cout << "Compression: " << (comp_info ? comp_info->name : "UNKNOWN") << std::endl;
    if (!comp_info->lossless) {
        std::cout << "Quality: " << quality << std::endl;
    }
    
    // Load input with GDAL
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* image = GDALImageLoader::to_heif_image(gdal_data, output_bit_depth);
    
    // Create context
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(compression);
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error(error, "get encoder");
    
    // Set compression parameters
    if (!comp_info->lossless && heif_format != heif_compression_uncompressed) {
        heif_encoder_set_lossy_quality(encoder, quality);
    } else if (comp_info->lossless) {
        heif_encoder_set_lossless(encoder, true);
    }
    
    // Set encoding options
    heif_encoding_options* options = heif_encoding_options_alloc();
    
    // Handle UNCI compression methods
    if (heif_format == heif_compression_uncompressed) {
        options->unci_compression = to_unci_compression(compression);
    }
    
    // Encode
    heif_image_handle* handle = nullptr;
    error = heif_context_encode_image(ctx, image, encoder, options, &handle);
    check_error(error, "encode image");
    
    // Set as primary
    heif_context_set_primary_image(ctx, handle);
    
    // Write to file
    error = heif_context_write_to_file(ctx, output_file);
    check_error(error, "write to file");
    
    std::cout << "Successfully encoded: " << output_file << std::endl;
    
    // Analyze the output file
    HEIFFileAnalyzer::analyze(output_file);
    
    // Cleanup
    heif_image_handle_release(handle);
    heif_encoding_options_free(options);
    heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(image);
}

std::cout << "Non-tiled encoding function defined." << std::endl;

#endif // HEIF_ENCODE_NONTILED_H

## Enhanced Tiling Support with All Schemes

This section adds support for:
1. TILI (tiled image) encoding
2. Explicit choice between grid/unci/tili schemes
3. All compression algorithms with any tiling scheme
4. Pyramids/overviews with configurable levels and consistent tiling

In [ ]:
#ifndef HEIF_TILING_SCHEME_H
#define HEIF_TILING_SCHEME_H

// Tiling scheme enumeration
enum class TilingScheme : uint8_t {
    Grid,     // Standard grid layout (works with all compressions)
    UNCI,     // Uncompressed with optional compression (ISO 23001-17)
    TILI      // Tiled image (experimental)
};

struct TilingSchemeInfo {
    TilingScheme scheme;
    std::string_view name;
    std::string_view description;
    bool experimental;
    bool supports_all_compressions;
};

constexpr inline std::array<TilingSchemeInfo, 3> TILING_SCHEMES = {{
    {TilingScheme::Grid, "grid", "Standard grid layout", false, true},
    {TilingScheme::UNCI, "unci", "Uncompressed tiling (ISO 23001-17)", false, false},
    {TilingScheme::TILI, "tili", "Tiled image (experimental)", true, true}
}};

inline std::string_view get_tiling_scheme_name(TilingScheme scheme) {
    for (const auto& info : TILING_SCHEMES) {
        if (info.scheme == scheme) {
            return info.name;
        }
    }
    return "unknown";
}

inline bool is_compression_supported_for_scheme(ExtendedCompressionType compression, 
                                               TilingScheme scheme) {
    if (scheme == TilingScheme::Grid || scheme == TilingScheme::TILI) {
        return true;  // Grid and TILI support all compressions
    }
    
    if (scheme == TilingScheme::UNCI) {
        // UNCI only supports uncompressed or specific compressions
        using enum ExtendedCompressionType;
        return (compression == None || compression == Uncompressed ||
                compression == DEFLATE || compression == ZLIB || 
                compression == Brotli);
    }
    
    return false;
}

void list_tiling_schemes() {
    std::cout << "\n=== Available Tiling Schemes ===" << std::endl;
    std::cout << std::left << std::setw(10) << "Scheme" 
              << std::setw(40) << "Description"
              << std::setw(15) << "Experimental"
              << std::setw(20) << "All Compressions" << std::endl;
    std::cout << std::string(85, '-') << std::endl;
    
    for (const auto& info : TILING_SCHEMES) {
        std::cout << std::left << std::setw(10) << info.name
                  << std::setw(40) << info.description
                  << std::setw(15) << (info.experimental ? "Yes" : "No")
                  << std::setw(20) << (info.supports_all_compressions ? "Yes" : "Limited")
                  << std::endl;
    }
}

std::cout << "Tiling schemes defined." << std::endl;

#endif // HEIF_TILING_SCHEME_H

# Working Point

## Pad Image to Match Tile Grid

### Padding Metadata

In [ ]:
#ifndef HEIF_PADDING_METADATA_TPAD_H
#define HEIF_PADDING_METADATA_TPAD_H

#include <vector>
#include <cstring>
#include <optional>

// Compact padding metadata - uses 'tpad' (Tile Padding) magic
// Only stores original dimensions per level (24 bytes fixed)
struct CompactPaddingMetadata {
    // Header (24 bytes) - minimal storage
    uint32_t magic;              // 'tpad' (0x74706164) for validation
    uint16_t version;            // Format version (1)
    uint16_t flags;              // Reserved flags
    
    uint32_t original_width;     // Original image width (before padding)
    uint32_t original_height;    // Original image height (before padding)
    uint16_t tile_width;         // Tile width
    uint16_t tile_height;        // Tile height
    uint16_t padding_value;      // Value used for padding
    uint16_t reserved;           // Alignment padding
    
    // Computed properties (not stored)
    
    uint16_t get_num_cols() const {
        return (original_width + tile_width - 1) / tile_width;
    }
    
    uint16_t get_num_rows() const {
        return (original_height + tile_height - 1) / tile_height;
    }
    
    uint32_t get_padded_width() const {
        return get_num_cols() * tile_width;
    }
    
    uint32_t get_padded_height() const {
        return get_num_rows() * tile_height;
    }
    
    size_t get_total_tiles() const {
        return static_cast<size_t>(get_num_cols()) * get_num_rows();
    }
    
    // Per-tile information computed on demand
    struct TileDataRegion {
        uint16_t data_x;        // X offset of data within tile (always 0)
        uint16_t data_y;        // Y offset of data within tile (always 0)
        uint16_t data_width;    // Actual data width in this tile
        uint16_t data_height;   // Actual data height in this tile
        bool has_data;          // False if tile is fully padded
        
        bool is_fully_padded() const {
            return !has_data || data_width == 0 || data_height == 0;
        }
        
        bool is_full_tile(uint16_t tile_width, uint16_t tile_height) const {
            return has_data && 
                   data_width == tile_width && 
                   data_height == tile_height;
        }
        
        bool needs_cropping(uint16_t tile_width, uint16_t tile_height) const {
            return has_data && 
                   (data_width < tile_width || data_height < tile_height);
        }
    };
    
    // Compute tile data region for specific tile
    TileDataRegion get_tile_region(uint16_t col, uint16_t row) const {
        TileDataRegion region;
        region.data_x = 0;
        region.data_y = 0;
        
        // Calculate tile position in image space
        uint32_t tile_left = col * tile_width;
        uint32_t tile_top = row * tile_height;
        
        // Check if tile is completely outside original bounds
        if (tile_left >= original_width || tile_top >= original_height) {
            region.has_data = false;
            region.data_width = 0;
            region.data_height = 0;
            return region;
        }
        
        region.has_data = true;
        
        // Calculate actual data width in this tile
        uint32_t tile_right = tile_left + tile_width;
        if (tile_right > original_width) {
            region.data_width = original_width - tile_left;
        } else {
            region.data_width = tile_width;
        }
        
        // Calculate actual data height in this tile
        uint32_t tile_bottom = tile_top + tile_height;
        if (tile_bottom > original_height) {
            region.data_height = original_height - tile_top;
        } else {
            region.data_height = tile_height;
        }
        
        return region;
    }
    
    // Get tile statistics (computed)
    struct TileStatistics {
        size_t total_tiles;
        size_t full_tiles;
        size_t partial_tiles;
        size_t fully_padded_tiles;
        
        void print() const {
            std::cout << "\n=== Tile Statistics (Computed) ===" << std::endl;
            std::cout << "Total tiles: " << total_tiles << std::endl;
            std::cout << "  Full data tiles: " << full_tiles 
                      << " (" << (full_tiles * 100.0 / total_tiles) << "%)" << std::endl;
            std::cout << "  Partially padded: " << partial_tiles 
                      << " (" << (partial_tiles * 100.0 / total_tiles) << "%)" << std::endl;
            std::cout << "  Fully padded: " << fully_padded_tiles 
                      << " (" << (fully_padded_tiles * 100.0 / total_tiles) << "%)" << std::endl;
        }
    };
    
    TileStatistics compute_statistics() const {
        TileStatistics stats{};
        stats.total_tiles = get_total_tiles();
        
        uint16_t num_cols = get_num_cols();
        uint16_t num_rows = get_num_rows();
        
        for (uint16_t row = 0; row < num_rows; row++) {
            for (uint16_t col = 0; col < num_cols; col++) {
                auto region = get_tile_region(col, row);
                
                if (region.is_fully_padded()) {
                    stats.fully_padded_tiles++;
                } else if (region.is_full_tile(tile_width, tile_height)) {
                    stats.full_tiles++;
                } else {
                    stats.partial_tiles++;
                }
            }
        }
        
        return stats;
    }
    
    // Serialize to binary (only 24 bytes!)
    std::vector<uint8_t> serialize() const {
        std::vector<uint8_t> buffer(24);
        uint8_t* ptr = buffer.data();
        
        memcpy(ptr, &magic, 4); ptr += 4;
        memcpy(ptr, &version, 2); ptr += 2;
        memcpy(ptr, &flags, 2); ptr += 2;
        memcpy(ptr, &original_width, 4); ptr += 4;
        memcpy(ptr, &original_height, 4); ptr += 4;
        memcpy(ptr, &tile_width, 2); ptr += 2;
        memcpy(ptr, &tile_height, 2); ptr += 2;
        memcpy(ptr, &padding_value, 2); ptr += 2;
        memcpy(ptr, &reserved, 2); ptr += 2;
        
        return buffer;
    }
    
    // Deserialize from binary
    static std::optional<CompactPaddingMetadata> deserialize(const uint8_t* data, size_t size) {
        if (size < 24) {
            return std::nullopt;
        }
        
        CompactPaddingMetadata meta;
        const uint8_t* ptr = data;
        
        memcpy(&meta.magic, ptr, 4); ptr += 4;
        if (meta.magic != 0x74706164) { // 'tpad'
            return std::nullopt;
        }
        
        memcpy(&meta.version, ptr, 2); ptr += 2;
        if (meta.version != 1) {
            return std::nullopt;
        }
        
        memcpy(&meta.flags, ptr, 2); ptr += 2;
        memcpy(&meta.original_width, ptr, 4); ptr += 4;
        memcpy(&meta.original_height, ptr, 4); ptr += 4;
        memcpy(&meta.tile_width, ptr, 2); ptr += 2;
        memcpy(&meta.tile_height, ptr, 2); ptr += 2;
        memcpy(&meta.padding_value, ptr, 2); ptr += 2;
        memcpy(&meta.reserved, ptr, 2); ptr += 2;
        
        return meta;
    }
    
    // Create from image dimensions
    static CompactPaddingMetadata create(
        uint32_t orig_width, uint32_t orig_height,
        uint16_t tile_w, uint16_t tile_h,
        uint16_t pad_value = 0) {
        
        CompactPaddingMetadata meta;
        meta.magic = 0x74706164;  // 'tpad'
        meta.version = 1;
        meta.flags = 0;
        meta.original_width = orig_width;
        meta.original_height = orig_height;
        meta.tile_width = tile_w;
        meta.tile_height = tile_h;
        meta.padding_value = pad_value;
        meta.reserved = 0;
        
        return meta;
    }
    
    void print() const {
        std::cout << "\n=== Tile Padding Metadata ('tpad') ===" << std::endl;
        std::cout << "Storage size: 24 bytes (fixed)" << std::endl;
        std::cout << "Magic: 'tpad' (0x" << std::hex << magic << std::dec << ")" << std::endl;
        std::cout << "\nStored values:" << std::endl;
        std::cout << "  Original size: " << original_width << "x" << original_height << std::endl;
        std::cout << "  Tile size: " << tile_width << "x" << tile_height << std::endl;
        std::cout << "  Padding value: " << padding_value << std::endl;
        
        std::cout << "\nComputed values:" << std::endl;
        std::cout << "  Padded size: " << get_padded_width() << "x" << get_padded_height() << std::endl;
        std::cout << "  Grid: " << get_num_cols() << "x" << get_num_rows() << " tiles" << std::endl;
        std::cout << "  Total tiles: " << get_total_tiles() << std::endl;
        std::cout << "  Right padding: " << (get_padded_width() - original_width) << " pixels" << std::endl;
        std::cout << "  Bottom padding: " << (get_padded_height() - original_height) << " pixels" << std::endl;
    }
    
    void print_tile_map() const {
        std::cout << "\n=== Tile Padding Map (Computed) ===" << std::endl;
        std::cout << "Legend: F=full data, P=partial data, X=fully padded\n" << std::endl;
        
        uint16_t num_cols = get_num_cols();
        uint16_t num_rows = get_num_rows();
        
        for (uint16_t row = 0; row < num_rows; row++) {
            for (uint16_t col = 0; col < num_cols; col++) {
                auto region = get_tile_region(col, row);
                
                if (region.is_fully_padded()) {
                    std::cout << "X ";
                } else if (region.is_full_tile(tile_width, tile_height)) {
                    std::cout << "F ";
                } else {
                    std::cout << "P ";
                }
            }
            std::cout << std::endl;
        }
    }
    
    void print_tile_details() const {
        std::cout << "\n=== Detailed Tile Information (Computed) ===" << std::endl;
        std::cout << std::left 
                  << std::setw(10) << "Tile(c,r)" 
                  << std::setw(15) << "Data Region"
                  << std::setw(10) << "Status" << std::endl;
        std::cout << std::string(45, '-') << std::endl;
        
        uint16_t num_cols = get_num_cols();
        uint16_t num_rows = get_num_rows();
        
        for (uint16_t row = 0; row < num_rows; row++) {
            for (uint16_t col = 0; col < num_cols; col++) {
                auto region = get_tile_region(col, row);
                
                std::cout << std::left
                          << std::setw(10) << std::format("({},{})", col, row)
                          << std::setw(15) << std::format("{}x{}", 
                                                          region.data_width, 
                                                          region.data_height);
                
                if (region.is_fully_padded()) {
                    std::cout << "Fully padded";
                } else if (region.is_full_tile(tile_width, tile_height)) {
                    std::cout << "Full tile";
                } else {
                    uint16_t right_pad = tile_width - region.data_width;
                    uint16_t bottom_pad = tile_height - region.data_height;
                    std::cout << std::format("Partial (R:{} B:{})", right_pad, bottom_pad);
                }
                std::cout << std::endl;
            }
        }
    }
};

std::cout << "Tile padding metadata ('tpad') defined (24 bytes fixed size)." << std::endl;

#endif // HEIF_PADDING_METADATA_TPAD_H

### Padding Function

In [ ]:
#ifndef HEIF_IMAGE_PADDING_H
#define HEIF_IMAGE_PADDING_H

heif_image* pad_image_for_tiling(heif_image* source, 
                                 int tile_width, int tile_height,
                                 uint16_t padding_value = 0) {  // Changed to uint16_t to support up to 16-bit
    int source_width = heif_image_get_primary_width(source);
    int source_height = heif_image_get_primary_height(source);
    
    // Calculate padded dimensions (round up to next tile boundary)
    int num_cols = (source_width + tile_width - 1) / tile_width;
    int num_rows = (source_height + tile_height - 1) / tile_height;
    int padded_width = num_cols * tile_width;
    int padded_height = num_rows * tile_height;
    
    // If already aligned, return as-is
    if (source_width == padded_width && source_height == padded_height) {
        return source;
    }
    
    std::cout << "    Padding image: " << source_width << "x" << source_height 
              << " → " << padded_width << "x" << padded_height 
              << " (padding value: " << padding_value << ")" << std::endl;
    
    // Create padded image
    heif_colorspace colorspace = heif_image_get_colorspace(source);
    heif_chroma chroma = heif_image_get_chroma_format(source);
    
    heif_image* padded = nullptr;
    heif_error error = heif_image_create(padded_width, padded_height, 
                                        colorspace, chroma, &padded);
    check_error_simple(error, "create padded image");
    
    // Get all channels
    std::vector<heif_channel> channels;
    if (heif_image_has_channel(source, heif_channel_Y)) channels.push_back(heif_channel_Y);
    if (heif_image_has_channel(source, heif_channel_Cb)) channels.push_back(heif_channel_Cb);
    if (heif_image_has_channel(source, heif_channel_Cr)) channels.push_back(heif_channel_Cr);
    if (heif_image_has_channel(source, heif_channel_R)) channels.push_back(heif_channel_R);
    if (heif_image_has_channel(source, heif_channel_G)) channels.push_back(heif_channel_G);
    if (heif_image_has_channel(source, heif_channel_B)) channels.push_back(heif_channel_B);
    if (heif_image_has_channel(source, heif_channel_Alpha)) channels.push_back(heif_channel_Alpha);
    
    // Copy and pad each channel
    for (auto channel : channels) {
        int bits = heif_image_get_bits_per_pixel(source, channel);
        heif_image_add_plane(padded, channel, padded_width, padded_height, bits);
        
        int src_stride, dst_stride;
        const uint8_t* src_data = heif_image_get_plane_readonly(source, channel, &src_stride);
        uint8_t* dst_data = heif_image_get_plane(padded, channel, &dst_stride);
        
        std::cout << "      Channel bits per pixel: " << bits << std::endl;
        
        if (bits <= 8) {
            // 8-bit data: use uint8_t
            uint8_t pad_val_8 = static_cast<uint8_t>(padding_value & 0xFF);
            
            // Copy source rows
            for (int y = 0; y < source_height; y++) {
                memcpy(dst_data + y * dst_stride,
                       src_data + y * src_stride,
                       source_width);
                
                // Pad right edge with padding_value
                if (source_width < padded_width) {
                    memset(dst_data + y * dst_stride + source_width, 
                           pad_val_8, 
                           padded_width - source_width);
                }
            }
            
            // Pad bottom rows with padding_value
            for (int y = source_height; y < padded_height; y++) {
                memset(dst_data + y * dst_stride,
                       pad_val_8,
                       padded_width);
            }
        } else {
            // 16-bit data: use uint16_t
            const uint16_t* src_data_16 = reinterpret_cast<const uint16_t*>(src_data);
            uint16_t* dst_data_16 = reinterpret_cast<uint16_t*>(dst_data);
            int src_stride_16 = src_stride / 2;
            int dst_stride_16 = dst_stride / 2;
            
            // Clamp padding value to max for bit depth
            uint16_t max_val = (1 << bits) - 1;
            uint16_t pad_val_16 = std::min(padding_value, max_val);
            
            // Copy source rows
            for (int y = 0; y < source_height; y++) {
                // Copy existing data
                for (int x = 0; x < source_width; x++) {
                    dst_data_16[y * dst_stride_16 + x] = src_data_16[y * src_stride_16 + x];
                }
                
                // Pad right edge
                if (source_width < padded_width) {
                    for (int x = source_width; x < padded_width; x++) {
                        dst_data_16[y * dst_stride_16 + x] = pad_val_16;
                    }
                }
            }
            
            // Pad bottom rows
            for (int y = source_height; y < padded_height; y++) {
                for (int x = 0; x < padded_width; x++) {
                    dst_data_16[y * dst_stride_16 + x] = pad_val_16;
                }
            }
        }
    }
    
    return padded;
}

// Check if image needs padding for tiling scheme
bool needs_padding_for_scheme(TilingScheme scheme, int width, int height, 
                              int tile_width, int tile_height) {
    if (scheme == TilingScheme::Grid) {
        return false;  // Grid handles partial tiles
    }
    
    // UNCI and TILI require exact divisibility
    return (width % tile_width != 0) || (height % tile_height != 0);
}

#endif // HEIF_IMAGE_PADDING_H

### Writing Padding info into HEIF

In [ ]:
#ifndef HEIF_WRITE_COMPACT_PADDING_FIXED_H
#define HEIF_WRITE_COMPACT_PADDING_FIXED_H

enum class PaddingMetadataFormat {
    TPAD_BINARY,    // 24-byte binary (default, most efficient)
    XMP_TEXT,       // XMP format (for compatibility)
    BOTH            // Store both formats
};

// Also add to single-image tiling config
struct TilingConfig {
    bool store_padding_metadata;
    PaddingMetadataFormat padding_format;
    
    TilingConfig()
        : store_padding_metadata(true)
        , padding_format(PaddingMetadataFormat::TPAD_BINARY) {}
};

// Write compact padding metadata (RECOMMENDED - only 24 bytes)
// FIXED: heif_context_add_generic_metadata takes 6 arguments
void write_compact_padding_metadata(heif_context* ctx, 
                                    heif_image_handle* handle,
                                    const CompactPaddingMetadata& meta,
                                    int level = 0) {
    
    std::cout << "  Writing compact padding metadata for level " << level 
              << " (24 bytes)..." << std::endl;
    
    auto binary_data = meta.serialize();
    
    // CORRECT: 6 arguments - ctx, handle, data, size, item_type, content_type
    heif_item_id metadata_id;
    heif_error error = heif_context_add_generic_metadata(
        ctx,
        handle,
        binary_data.data(),
        static_cast<int>(binary_data.size()),
        "cpad",  // item_type
        "application/octet-stream"  // content_type
    );
    
    if (error.code == heif_error_Ok) {
        std::cout << "  ✓ Compact padding metadata written for level " << level 
                  << " (24 bytes)" << std::endl;
        meta.print();
    } else {
        std::cerr << "  ⚠ Failed to write padding metadata: " 
                  << error.message << std::endl;
    }
}

// Alternative: Write as Exif metadata (also widely supported)
void write_compact_padding_as_exif(heif_context* ctx,
                                   heif_image_handle* handle,
                                   const CompactPaddingMetadata& meta,
                                   int level = 0) {
    
    std::cout << "  Writing padding as Exif metadata for level " << level << std::endl;
    
    auto binary_data = meta.serialize();
    
    // Exif data needs to start with "Exif\0\0" header + TIFF header
    // For simplicity, we'll use a custom Exif tag in the user comment
    std::vector<uint8_t> exif_data;
    
    // Add minimal TIFF header (we'll store in UserComment)
    // This is a simplified approach - full Exif is more complex
    const char* exif_header = "Exif\0\0";
    exif_data.insert(exif_data.end(), exif_header, exif_header + 6);
    
    // Add our binary data
    exif_data.insert(exif_data.end(), binary_data.begin(), binary_data.end());
    
    heif_error error = heif_context_add_exif_metadata(
        ctx,
        handle,
        exif_data.data(),
        static_cast<int>(exif_data.size())
    );
    
    if (error.code == heif_error_Ok) {
        std::cout << "  ✓ Padding metadata written as Exif for level " << level << std::endl;
    } else {
        std::cerr << "  ⚠ Failed to write Exif metadata: " 
                  << error.message << std::endl;
    }
}

// Simple alternative: Store as a custom string in XMP
void write_compact_padding_as_simple_xmp(heif_context* ctx,
                                        heif_image_handle* handle,
                                        const CompactPaddingMetadata& meta,
                                        int level = 0) {
    
    // Create simple XMP with binary data as base64 or hex string
    std::string hex_data;
    auto binary_data = meta.serialize();
    
    // Convert to hex string
    for (uint8_t byte : binary_data) {
        char buf[3];
        snprintf(buf, sizeof(buf), "%02x", byte);
        hex_data += buf;
    }
    
    std::string xmp_data = std::format(R"(<?xml version="1.0"?>
<x:xmpmeta xmlns:x="adobe:ns:meta/">
  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
    <rdf:Description rdf:about=""
        xmlns:heif="http://ns.example.com/heif/1.0/">
      <heif:PaddingLevel>{}</heif:PaddingLevel>
      <heif:OriginalWidth>{}</heif:OriginalWidth>
      <heif:OriginalHeight>{}</heif:OriginalHeight>
      <heif:TileWidth>{}</heif:TileWidth>
      <heif:TileHeight>{}</heif:TileHeight>
      <heif:PaddingValue>{}</heif:PaddingValue>
      <heif:BinaryMetadata>{}</heif:BinaryMetadata>
    </rdf:Description>
  </rdf:RDF>
</x:xmpmeta>)",
        level,
        meta.original_width, meta.original_height,
        meta.tile_width, meta.tile_height,
        meta.padding_value,
        hex_data);
    
    heif_error error = heif_context_add_XMP_metadata(
        ctx, 
        handle,
        xmp_data.c_str(),
        static_cast<int>(xmp_data.size())
    );
    
    if (error.code == heif_error_Ok) {
        std::cout << "  ✓ XMP padding metadata written for level " << level 
                  << " (" << xmp_data.size() << " bytes)" << std::endl;
    } else {
        std::cerr << "  ⚠ Failed to write XMP metadata: " 
                  << error.message << std::endl;
    }
}

std::cout << "Fixed padding metadata writers defined." << std::endl;

#endif // HEIF_WRITE_COMPACT_PADDING_FIXED_H

In [ ]:
#ifndef HEIF_WRITE_METADATA_TPAD_H
#define HEIF_WRITE_METADATA_TPAD_H

// Quick reference for magic values:
// 
// Magic: 'tpad' = 0x74706164
//   't' = 0x74
//   'p' = 0x70
//   'a' = 0x61
//   'd' = 0x64
//
// To verify in hex dump: look for bytes "74 70 61 64" (big-endian)
// or "64 61 70 74" (little-endian on x86)

// Write tile padding metadata using 'tpad' item type
void write_tpad_metadata(heif_context* ctx, 
                         heif_image_handle* handle,
                         const CompactPaddingMetadata& meta,
                         int level = 0) {
    
    std::cout << "  Writing tile padding metadata ('tpad') for level " << level 
              << " (24 bytes)..." << std::endl;
    
    auto binary_data = meta.serialize();
    
    // heif_context_add_generic_metadata signature:
    // (ctx, handle, data, size, item_type, content_type)
    heif_error error = heif_context_add_generic_metadata(
        ctx,
        handle,
        binary_data.data(),
        static_cast<int>(binary_data.size()),
        "tpad",                       // item_type - Tile Padding
        "application/octet-stream"    // content_type
    );
    
    if (error.code == heif_error_Ok) {
        std::cout << "  ✓ Tile padding metadata written for level " << level 
                  << " (24 bytes, magic: 'tpad')" << std::endl;
    } else {
        std::cerr << "  ⚠ Failed to write tile padding metadata: " 
                  << error.message << std::endl;
    }
}

std::cout << "Tile padding metadata writer ('tpad') defined." << std::endl;

#endif // HEIF_WRITE_METADATA_TPAD_H

### Padding info reader

In [ ]:
#ifndef HEIF_READ_TPAD_METADATA_H
#define HEIF_READ_TPAD_METADATA_H

// When implementing your own reader, search for:
// Magic bytes: 0x74 0x70 0x61 0x64  (big-endian 'tpad')
// 
// Full 24-byte structure:
// Offset  Size  Field
// 0       4     magic (0x74706164 = 'tpad')
// 4       2     version (1)
// 6       2     flags (0)
// 8       4     original_width
// 12      4     original_height
// 16      2     tile_width
// 18      2     tile_height
// 20      2     padding_value
// 22      2     reserved (0)

class TpadMetadataReader {
public:
    // Read tile padding metadata ('tpad')
    static std::optional<std::vector<CompactPaddingMetadata>> 
    read_all_levels(const char* filename) {
        
        heif_context* ctx = heif_context_alloc();
        heif_error error = heif_context_read_from_file(ctx, filename, nullptr);
        
        if (error.code != heif_error_Ok) {
            std::cerr << "Cannot open file: " << error.message << std::endl;
            heif_context_free(ctx);
            return std::nullopt;
        }
        
        std::vector<CompactPaddingMetadata> all_metadata;
        
        int num_images = heif_context_get_number_of_top_level_images(ctx);
        std::vector<heif_item_id> ids(num_images);
        heif_context_get_list_of_top_level_image_IDs(ctx, ids.data(), num_images);
        
        std::cout << "Searching for 'tpad' metadata in " << num_images << " image(s)..." << std::endl;
        
        for (int i = 0; i < num_images; i++) {
            heif_image_handle* handle;
            error = heif_context_get_image_handle(ctx, ids[i], &handle);
            if (error.code != heif_error_Ok) continue;
            
            // Look for "tpad" metadata (tile padding)
            heif_item_id tpad_ids[10];
            int num_tpad = heif_image_handle_get_list_of_metadata_block_IDs(
                handle, "tpad", tpad_ids, 10);
            
            if (num_tpad > 0) {
                std::cout << "  Found " << num_tpad << " 'tpad' metadata block(s) in image " << i << std::endl;
            }
            
            for (int j = 0; j < num_tpad; j++) {
                size_t metadata_size = heif_image_handle_get_metadata_size(
                    handle, tpad_ids[j]);
                
                std::cout << "    Block " << j << ": " << metadata_size << " bytes" << std::endl;
                
                if (metadata_size == 24) {  // Verify size
                    std::vector<uint8_t> metadata_buffer(metadata_size);
                    error = heif_image_handle_get_metadata(
                        handle, tpad_ids[j], metadata_buffer.data());
                    
                    if (error.code == heif_error_Ok) {
                        // Verify magic before deserializing
                        uint32_t magic;
                        memcpy(&magic, metadata_buffer.data(), 4);
                        
                        if (magic == 0x74706164) {  // 'tpad'
                            auto meta = CompactPaddingMetadata::deserialize(
                                metadata_buffer.data(), metadata_size);
                            
                            if (meta) {
                                std::cout << "    ✓ Valid 'tpad' metadata found" << std::endl;
                                all_metadata.push_back(*meta);
                            } else {
                                std::cout << "    ✗ Failed to deserialize 'tpad' metadata" << std::endl;
                            }
                        } else {
                            std::cout << "    ✗ Invalid magic: 0x" << std::hex << magic << std::dec << std::endl;
                        }
                    }
                } else {
                    std::cout << "    ✗ Wrong size (expected 24 bytes)" << std::endl;
                }
            }
            
            heif_image_handle_release(handle);
        }
        
        heif_context_free(ctx);
        
        if (all_metadata.empty()) {
            std::cout << "No 'tpad' metadata found in file" << std::endl;
            return std::nullopt;
        }
        
        std::cout << "✓ Successfully read " << all_metadata.size() << " 'tpad' metadata block(s)" << std::endl;
        return all_metadata;
    }
    
    // Alternative: Extract tile data region by copying pixels manually
    // (No heif_image_crop used)
    static heif_image* extract_unpadded_tile_manual(
        heif_image* padded_tile,
        const CompactPaddingMetadata::TileDataRegion& region,
        uint16_t tile_width,
        uint16_t tile_height) {
        
        if (region.is_fully_padded()) {
            return nullptr;
        }
        
        if (region.is_full_tile(tile_width, tile_height)) {
            return padded_tile;  // No cropping needed
        }
        
        // Create new image with actual data dimensions
        heif_image* cropped;
        heif_error error = heif_image_create(
            region.data_width, region.data_height,
            heif_image_get_colorspace(padded_tile),
            heif_image_get_chroma_format(padded_tile),
            &cropped);
        
        if (error.code != heif_error_Ok) {
            std::cerr << "Failed to create cropped image: " << error.message << std::endl;
            return nullptr;
        }
        
        // Copy each channel
        std::vector<heif_channel> channels = {
            heif_channel_Y, heif_channel_Cb, heif_channel_Cr,
            heif_channel_R, heif_channel_G, heif_channel_B, 
            heif_channel_Alpha
        };
        
        for (auto channel : channels) {
            if (!heif_image_has_channel(padded_tile, channel)) continue;
            
            int bits = heif_image_get_bits_per_pixel(padded_tile, channel);
            heif_image_add_plane(cropped, channel, 
                               region.data_width, region.data_height, bits);
            
            int src_stride, dst_stride;
            const uint8_t* src_data = heif_image_get_plane_readonly(
                padded_tile, channel, &src_stride);
            uint8_t* dst_data = heif_image_get_plane(cropped, channel, &dst_stride);
            
            if (bits <= 8) {
                // 8-bit data
                for (int y = 0; y < region.data_height; y++) {
                    const uint8_t* src_row = src_data + 
                        (y + region.data_y) * src_stride + region.data_x;
                    uint8_t* dst_row = dst_data + y * dst_stride;
                    memcpy(dst_row, src_row, region.data_width);
                }
            } else {
                // 16-bit data
                const uint16_t* src_16 = reinterpret_cast<const uint16_t*>(src_data);
                uint16_t* dst_16 = reinterpret_cast<uint16_t*>(dst_data);
                int src_stride_16 = src_stride / 2;
                int dst_stride_16 = dst_stride / 2;
                
                for (int y = 0; y < region.data_height; y++) {
                    const uint16_t* src_row = src_16 + 
                        (y + region.data_y) * src_stride_16 + region.data_x;
                    uint16_t* dst_row = dst_16 + y * dst_stride_16;
                    memcpy(dst_row, src_row, region.data_width * 2);
                }
            }
        }
        
        return cropped;
    }
    
    // Display all found metadata
    static void display_all_metadata(const char* filename) {
        std::cout << "\n=== Reading Tile Padding Metadata ('tpad') ===" << std::endl;
        std::cout << "File: " << filename << std::endl;
        
        auto metadata_list = read_all_levels(filename);
        
        if (!metadata_list || metadata_list->empty()) {
            std::cout << "\nNo 'tpad' metadata found in file" << std::endl;
            return;
        }
        
        std::cout << "\n" << std::string(60, '=') << std::endl;
        std::cout << "Found " << metadata_list->size() << " level(s) with padding metadata" << std::endl;
        std::cout << std::string(60, '=') << std::endl;
        
        for (size_t i = 0; i < metadata_list->size(); i++) {
            std::cout << "\n--- Level " << i << " ---" << std::endl;
            (*metadata_list)[i].print();
            (*metadata_list)[i].print_tile_map();
            
            auto stats = (*metadata_list)[i].compute_statistics();
            stats.print();
        }
    }
};

// Process tiles using 'tpad' metadata
void process_tiles_with_tpad(const char* filename) {
    std::cout << "\n=== Processing Tiles with 'tpad' Metadata ===" << std::endl;
    
    auto metadata_list = TpadMetadataReader::read_all_levels(filename);
    if (!metadata_list) {
        std::cerr << "No 'tpad' metadata found" << std::endl;
        return;
    }
    
    const auto& meta = (*metadata_list)[0];
    meta.print();
    meta.print_tile_map();
    meta.print_tile_details();
    
    auto stats = meta.compute_statistics();
    stats.print();
    
    // Open HEIF file
    heif_context* ctx = heif_context_alloc();
    heif_error error = heif_context_read_from_file(ctx, filename, nullptr);
    check_error_simple(error, "read file");
    
    heif_image_handle* handle;
    error = heif_context_get_primary_image_handle(ctx, &handle);
    check_error_simple(error, "get handle");
    
    heif_image* full_image;
    error = heif_decode_image(handle, &full_image, heif_colorspace_undefined,
                              heif_chroma_undefined, nullptr);
    
    if (error.code != heif_error_Ok) {
        std::cout << "⚠ Cannot decode primary image: " << error.message << std::endl;
        std::cout << "  (This is expected for TILI images)" << std::endl;
        heif_image_handle_release(handle);
        heif_context_free(ctx);
        return;
    }
    
    // Process tiles
    int tiles_processed = 0;
    int tiles_skipped = 0;
    int tiles_cropped = 0;
    
    uint16_t num_cols = meta.get_num_cols();
    uint16_t num_rows = meta.get_num_rows();
    
    for (uint16_t row = 0; row < num_rows; row++) {
        for (uint16_t col = 0; col < num_cols; col++) {
            // Compute tile region on-the-fly
            auto region = meta.get_tile_region(col, row);
            
            if (region.is_fully_padded()) {
                tiles_skipped++;
                continue;
            }
            
            int tile_x = col * meta.tile_width;
            int tile_y = row * meta.tile_height;
            
            heif_image* tile_image;
            error = heif_image_extract_area(full_image,
                                           tile_x, tile_y,
                                           meta.tile_width, meta.tile_height,
                                           nullptr, &tile_image);
            
            if (error.code == heif_error_Ok) {
                // Use manual extraction (no heif_image_crop)
                heif_image* unpadded = TpadMetadataReader::extract_unpadded_tile_manual(
                    tile_image, region, meta.tile_width, meta.tile_height);
                
                if (unpadded) {
                    bool was_cropped = (unpadded != tile_image);
                    if (was_cropped) tiles_cropped++;
                    
                    std::cout << std::format("Tile ({},{}) - {}x{} {}",
                                           col, row, 
                                           region.data_width, region.data_height,
                                           was_cropped ? "[cropped]" : "[full]") 
                              << std::endl;
                    
                    tiles_processed++;
                    
                    // Release cropped image if it's different
                    if (unpadded != tile_image) {
                        heif_image_release(unpadded);
                    }
                }
                
                heif_image_release(tile_image);
            }
        }
    }
    
    std::cout << "\n=== Processing Summary ===" << std::endl;
    std::cout << "Tiles processed: " << tiles_processed 
              << " (full: " << (tiles_processed - tiles_cropped) 
              << ", cropped: " << tiles_cropped << ")" << std::endl;
    std::cout << "Tiles skipped: " << tiles_skipped << std::endl;
    
    heif_image_release(full_image);
    heif_image_handle_release(handle);
    heif_context_free(ctx);
}

std::cout << "Tile padding ('tpad') metadata reader defined." << std::endl;

#endif // HEIF_READ_TPAD_METADATA_H

### Multi-level tile padding info

In [ ]:
#ifndef HEIF_MULTILEVEL_TPAD_H
#define HEIF_MULTILEVEL_TPAD_H

// Multi-level padding metadata container
struct MultiLevelPaddingMetadata {
    uint32_t magic;              // 'mtpd' (Multi-level Tile PaDding)
    uint16_t version;            // Format version (1)
    uint16_t num_levels;         // Number of pyramid levels
    
    // Array of level metadata (24 bytes each)
    std::vector<CompactPaddingMetadata> levels;
    
    // Serialize to binary
    std::vector<uint8_t> serialize() const {
        // Header: 8 bytes
        // Levels: 24 bytes × num_levels
        size_t total_size = 8 + (24 * levels.size());
        std::vector<uint8_t> buffer(total_size);
        uint8_t* ptr = buffer.data();
        
        // Header
        memcpy(ptr, &magic, 4); ptr += 4;
        memcpy(ptr, &version, 2); ptr += 2;
        memcpy(ptr, &num_levels, 2); ptr += 2;
        
        // Each level's metadata
        for (const auto& level : levels) {
            auto level_data = level.serialize();
            memcpy(ptr, level_data.data(), 24);
            ptr += 24;
        }
        
        return buffer;
    }
    
    // Deserialize from binary
    static std::optional<MultiLevelPaddingMetadata> deserialize(const uint8_t* data, size_t size) {
        if (size < 8) return std::nullopt;
        
        MultiLevelPaddingMetadata meta;
        const uint8_t* ptr = data;
        
        memcpy(&meta.magic, ptr, 4); ptr += 4;
        if (meta.magic != 0x6D747064) { // 'mtpd'
            return std::nullopt;
        }
        
        memcpy(&meta.version, ptr, 2); ptr += 2;
        if (meta.version != 1) return std::nullopt;
        
        memcpy(&meta.num_levels, ptr, 2); ptr += 2;
        
        // Validate size
        size_t expected_size = 8 + (24 * meta.num_levels);
        if (size != expected_size) return std::nullopt;
        
        // Read each level
        for (uint16_t i = 0; i < meta.num_levels; i++) {
            auto level_meta = CompactPaddingMetadata::deserialize(ptr, 24);
            if (!level_meta) return std::nullopt;
            meta.levels.push_back(*level_meta);
            ptr += 24;
        }
        
        return meta;
    }
    
    static MultiLevelPaddingMetadata create() {
        MultiLevelPaddingMetadata meta;
        meta.magic = 0x6D747064;  // 'mtpd'
        meta.version = 1;
        meta.num_levels = 0;
        return meta;
    }
    
    void add_level(const CompactPaddingMetadata& level) {
        levels.push_back(level);
        num_levels = static_cast<uint16_t>(levels.size());
    }
    
    void print() const {
        std::cout << "\n=== Multi-Level Tile Padding Metadata ('mtpd') ===" << std::endl;
        std::cout << "Total storage: " << (8 + 24 * num_levels) << " bytes" << std::endl;
        std::cout << "  Header: 8 bytes" << std::endl;
        std::cout << "  Levels: " << num_levels << " × 24 bytes = " << (24 * num_levels) << " bytes" << std::endl;
        std::cout << "\nPer-level details:" << std::endl;
        
        for (size_t i = 0; i < levels.size(); i++) {
            std::cout << "\n--- Level " << i << " ---" << std::endl;
            levels[i].print();
        }
    }
};

#endif // HEIF_MULTILEVEL_TPAD_H

### Write Multi-Level Padding Info

In [ ]:
#ifndef HEIF_WRITE_MULTILEVEL_TPAD_REVISED_H
#define HEIF_WRITE_MULTILEVEL_TPAD_REVISED_H

// Write multi-level padding metadata in different formats
void write_multilevel_padding_metadata(heif_context* ctx, 
                                       heif_image_handle* handle,
                                       const MultiLevelPaddingMetadata& meta,
                                       PaddingMetadataFormat format) {
    
    switch (format) {
        case PaddingMetadataFormat::TPAD_BINARY:
            // Write as 'mtpd' (multi-level tile padding) binary format
            std::cout << "  Writing multi-level padding metadata as TPAD Binary ('" 
                      << meta.num_levels << " levels, " 
                      << (8 + 24 * meta.num_levels) << " bytes total)..." << std::endl;
            
            {
                auto binary_data = meta.serialize();
                heif_error error = heif_context_add_generic_metadata(
                    ctx,
                    handle,
                    binary_data.data(),
                    static_cast<int>(binary_data.size()),
                    "mtpd",                       // Multi-level Tile PaDding
                    "application/octet-stream"
                );
                
                if (error.code == heif_error_Ok) {
                    std::cout << "  ✓ Multi-level TPAD Binary metadata written (" 
                              << binary_data.size() << " bytes)" << std::endl;
                } else {
                    std::cerr << "  ⚠ Failed to write TPAD Binary metadata: " 
                              << error.message << std::endl;
                }
            }
            break;
            
        case PaddingMetadataFormat::XMP_TEXT:
            // Write as XMP text format
            std::cout << "  Writing multi-level padding metadata as XMP..." << std::endl;
            
            {
                std::string xmp_data = R"(<?xml version="1.0"?>
<x:xmpmeta xmlns:x="adobe:ns:meta/">
  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
    <rdf:Description rdf:about=""
        xmlns:heif="http://ns.example.com/heif/1.0/">
      <heif:TilePadding>
        <rdf:Seq>
)";
                
                // Add each level
                for (size_t i = 0; i < meta.levels.size(); i++) {
                    const auto& level = meta.levels[i];
                    
                    // Convert binary to hex string
                    std::string hex_data;
                    auto binary_data = level.serialize();
                    for (uint8_t byte : binary_data) {
                        char buf[3];
                        snprintf(buf, sizeof(buf), "%02x", byte);
                        hex_data += buf;
                    }
                    
                    xmp_data += std::format(R"(          <rdf:li>
            <heif:Level>{}</heif:Level>
            <heif:OriginalWidth>{}</heif:OriginalWidth>
            <heif:OriginalHeight>{}</heif:OriginalHeight>
            <heif:TileWidth>{}</heif:TileWidth>
            <heif:TileHeight>{}</heif:TileHeight>
            <heif:PaddingValue>{}</heif:PaddingValue>
            <heif:BinaryData>{}</heif:BinaryData>
          </rdf:li>
)",
                        i,
                        level.original_width, level.original_height,
                        level.tile_width, level.tile_height,
                        level.padding_value,
                        hex_data);
                }
                
                xmp_data += R"(        </rdf:Seq>
      </heif:TilePadding>
    </rdf:Description>
  </rdf:RDF>
</x:xmpmeta>)";
                
                heif_error error = heif_context_add_XMP_metadata(
                    ctx, 
                    handle,
                    xmp_data.c_str(),
                    static_cast<int>(xmp_data.size())
                );
                
                if (error.code == heif_error_Ok) {
                    std::cout << "  ✓ XMP padding metadata written (" 
                              << xmp_data.size() << " bytes)" << std::endl;
                } else {
                    std::cerr << "  ⚠ Failed to write XMP metadata: " 
                              << error.message << std::endl;
                }
            }
            break;
            
        case PaddingMetadataFormat::BOTH:
            // Write both formats
            std::cout << "  Writing multi-level padding metadata in BOTH formats..." << std::endl;
            write_multilevel_padding_metadata(ctx, handle, meta, PaddingMetadataFormat::TPAD_BINARY);
            write_multilevel_padding_metadata(ctx, handle, meta, PaddingMetadataFormat::XMP_TEXT);
            break;
    }
}

#endif // HEIF_WRITE_MULTILEVEL_TPAD_REVISED_H

## Offset offset documentation

In [ ]:
#ifndef HEIF_OFFSET_TABLE_CONFIG_FINAL_H
#define HEIF_OFFSET_TABLE_CONFIG_FINAL_H

// Final, accurate OffsetTableConfig based on actual libheif API
struct OffsetTableConfig {
    uint8_t offset_field_length=0;    // Bits: 0, 32, 40, 48, 64
    uint8_t size_field_length=0;      // Bits: 0, 24, 32, 40, 48, 64
    uint8_t tiles_are_sequential=0;   // 0 = use tables, 1 = sequential
    
    enum class Preset : uint8_t {
        OffsetAndSize,      // Both offset and size tables
        OffsetOnly,         // Only offset table
        SizeOnly,           // Only size table
        Sequential,         // No tables (sequential)
        Default
    };
    
    // Constructors
    OffsetTableConfig() 
        : offset_field_length(32)
        , size_field_length(24)
        , tiles_are_sequential(0) {}
    
    explicit OffsetTableConfig(Preset preset) {
        switch (preset) {
            case Preset::OffsetAndSize:
                offset_field_length = 32;
                size_field_length = 24;
                tiles_are_sequential = 0;
                break;
            case Preset::OffsetOnly:
                offset_field_length = 32;
                size_field_length = 0;
                tiles_are_sequential = 1;
                break;
            case Preset::SizeOnly:
                offset_field_length = 0;
                size_field_length = 24;
                tiles_are_sequential = 1;
                break;
            case Preset::Sequential:
                offset_field_length = 0;
                size_field_length = 0;
                tiles_are_sequential = 1;
                break;
            case Preset::Default:
            default:
                offset_field_length = 32;
                size_field_length = 24;
                tiles_are_sequential = 0;
                break;
        }
    }
    
    OffsetTableConfig(uint8_t offset_bits, uint8_t size_bits, bool sequential = false)
        : offset_field_length(offset_bits)
        , size_field_length(size_bits)
        , tiles_are_sequential(sequential ? 1 : 0) {}
    
    // Validation
    bool is_valid() const {
        if (offset_field_length != 0 && offset_field_length != 32 && 
            offset_field_length != 40 && offset_field_length != 48 && 
            offset_field_length != 64) {
            return false;
        }
        
        if (size_field_length != 0 && size_field_length != 24 && 
            size_field_length != 32 && size_field_length != 40 && 
            size_field_length != 48 && size_field_length != 64) {
            return false;
        }
        
        return true;
    }
    
    // Check if directly controllable via API (ACCURATE VERSION)
    bool is_directly_controllable(TilingScheme scheme) const {
        switch (scheme) {
            case TilingScheme::Grid:
                // Grid: Uses iloc box, not directly controllable
                return false;
                
            case TilingScheme::UNCI:
                // UNCI: API does NOT expose offset/size fields yet
                // heif_unci_image_parameters has TODO for interleave/padding
                // icef box exists internally but not in public API
                return false;
                
            case TilingScheme::TILI:
                // TILI: Full API support via heif_tiled_image_parameters
                return true;
                
            default:
                return false;
        }
    }
    
    std::string api_status(TilingScheme scheme) const {
        switch (scheme) {
            case TilingScheme::Grid:
                return "❌ Grid: No API support\n"
                       "   • Uses file-level iloc box\n"
                       "   • Tiles stored as separate items\n"
                       "   • Offset/size not configurable per-grid";
                
            case TilingScheme::UNCI:
                return "❌ UNCI: Not yet in API (v1.x)\n"
                       "   • heif_unci_image_parameters incomplete\n"
                       "   • TODO: interleave type, padding\n"
                       "   • icef box exists internally but not exposed\n"
                       "   • May be added in future libheif version";
                
            case TilingScheme::TILI:
                return "✅ TILI: Full API support\n"
                       "   • heif_tiled_image_parameters.offset_field_length\n"
                       "   • heif_tiled_image_parameters.size_field_length\n"
                       "   • heif_tiled_image_parameters.tiles_are_sequential\n"
                       "   • Direct control over tile offset table";
                
            default:
                return "Unknown scheme";
        }
    }
    
    std::string description() const {
        if (tiles_are_sequential) {
            return "Sequential (no offset table)";
        }
        
        std::string desc;
        if (offset_field_length > 0 && size_field_length > 0) {
            desc = std::format("Offset+Size ({}+{} bits)", 
                             offset_field_length, size_field_length);
        } else if (offset_field_length > 0) {
            desc = std::format("Offset-only ({} bits)", offset_field_length);
        } else if (size_field_length > 0) {
            desc = std::format("Size-only ({} bits)", size_field_length);
        } else {
            desc = "No offset table";
        }
        return desc;
    }
    
    size_t table_entry_size_bytes() const {
        if (tiles_are_sequential) return 0;
        return (offset_field_length + size_field_length) / 8;
    }
};

// Print comprehensive comparison table
void print_offset_table_api_support() {
    std::cout << "\n╔════════════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Tile Offset/Size Table Configuration - API Support Status (libheif)  ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::cout << "\n┌─────────┬────────────────┬──────────────────────────────────────────────┐" << std::endl;
    std::cout << "│ Scheme  │ API Support    │ Details                                      │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ Grid    │ ❌ NO          │ • Uses file-level iloc box                   │" << std::endl;
    std::cout << "│         │                │ • Each tile = separate image item            │" << std::endl;
    std::cout << "│         │                │ • No per-grid offset configuration           │" << std::endl;
    std::cout << "│         │                │ • Best for: Standard multi-codec tiling      │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ UNCI    │ ❌ NO          │ • heif_unci_image_parameters incomplete      │" << std::endl;
    std::cout << "│         │ (current v1.x) │ • Has: tile_width, tile_height, compression  │" << std::endl;
    std::cout << "│         │                │ • Missing: offset/size table config          │" << std::endl;
    std::cout << "│         │ 🔧 FUTURE      │ • TODO comment: \"interleave type, padding\"   │" << std::endl;
    std::cout << "│         │                │ • icef box exists but not in public API      │" << std::endl;
    std::cout << "│         │                │ • May come in future libheif release         │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ TILI    │ ✅ YES         │ • heif_tiled_image_parameters provides:      │" << std::endl;
    std::cout << "│         │ FULL SUPPORT   │   - offset_field_length (0,32,40,48,64 bits) │" << std::endl;
    std::cout << "│         │                │   - size_field_length (0,24,32,40,48,64 bits)│" << std::endl;
    std::cout << "│         │                │   - tiles_are_sequential (0 or 1)            │" << std::endl;
    std::cout << "│         │                │ • Complete control over tile offset table    │" << std::endl;
    std::cout << "│         │                │ • Best for: Advanced offset table control    │" << std::endl;
    std::cout << "└─────────┴────────────────┴──────────────────────────────────────────────┘" << std::endl;
    
    std::cout << "\n📊 SUMMARY:" << std::endl;
    std::cout << "   • Only TILI provides direct offset/size table configuration" << std::endl;
    std::cout << "   • UNCI may add this in future (see TODO in heif_unci_image_parameters)" << std::endl;
    std::cout << "   • Grid uses standard HEIF item location mechanism" << std::endl;
    
    std::cout << "\n💡 RECOMMENDATION:" << std::endl;
    std::cout << "   Use TilingScheme::TILI for explicit offset/size table control" << std::endl;
    
    std::cout << "\n" << std::string(76, '=') << std::endl;
}

// List presets with accurate applicability
void list_offset_table_presets_with_support() {
    std::cout << "\n=== Offset Table Configuration Presets ===" << std::endl;
    std::cout << std::left << std::setw(20) << "Preset" 
              << std::setw(15) << "Offset Bits"
              << std::setw(15) << "Size Bits"
              << std::setw(15) << "Sequential"
              << "Description" << std::endl;
    std::cout << std::string(100, '-') << std::endl;
    
    auto print_preset = [](const char* name, OffsetTableConfig::Preset preset) {
        OffsetTableConfig config(preset);
        std::cout << std::left << std::setw(20) << name
                  << std::setw(15) << static_cast<int>(config.offset_field_length)
                  << std::setw(15) << static_cast<int>(config.size_field_length)
                  << std::setw(15) << (config.tiles_are_sequential ? "Yes" : "No")
                  << config.description() << std::endl;
    };
    
    print_preset("OffsetAndSize", OffsetTableConfig::Preset::OffsetAndSize);
    print_preset("OffsetOnly", OffsetTableConfig::Preset::OffsetOnly);
    print_preset("SizeOnly", OffsetTableConfig::Preset::SizeOnly);
    print_preset("Sequential", OffsetTableConfig::Preset::Sequential);
    print_preset("Default", OffsetTableConfig::Preset::Default);
    
    std::cout << "\n⚠️  IMPORTANT: These configurations are ONLY applied to TILI scheme!" << std::endl;
    std::cout << "   • Grid and UNCI do not support offset table configuration in current API" << std::endl;
    std::cout << "   • Config will be documented but not applied for Grid/UNCI" << std::endl;
}

std::cout << "Accurate offset table configuration structures defined." << std::endl;

#endif // HEIF_OFFSET_TABLE_CONFIG_FINAL_H

## Encoding function with offset configuration

In [ ]:
#ifndef HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H
#define HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H

// Universal tiled encoding function with offset table configuration
void encode_tiled(const char* input_file, 
                 const char* output_file,
                 int tile_width, 
                 int tile_height,
                 TilingScheme scheme,
                 ExtendedCompressionType compression,
                 int quality = 50,
                 int output_bit_depth = 8,
                 uint16_t padding_value = 0,
                 const OffsetTableConfig& offset_config = OffsetTableConfig(),
                 const TilingConfig& tiling_config = TilingConfig()) {
    
    std::cout << "\n=== Encoding Tiled Image ===" << std::endl;
    std::cout << "Tiling Scheme: " << get_tiling_scheme_name(scheme) << std::endl;
    std::cout << "Compression: " << get_compression_name(compression) << std::endl;
    std::cout << "Tile Size: " << tile_width << "x" << tile_height << std::endl;
    std::cout << "Padding Value: " << padding_value << std::endl;
    std::cout << "Offset Table Config: " << offset_config.description() << std::endl;

    // Show offset config status
    if (offset_config.is_directly_controllable(scheme)) {
        std::cout << "\n✅ Offset Table Config: WILL BE APPLIED" << std::endl;
        std::cout << "   " << offset_config.description() << std::endl;
    } else {
        std::cout << "\n⚠️  Offset Table Config: NOT APPLIED (not supported by scheme)" << std::endl;
        std::cout << "   Requested: " << offset_config.description() << std::endl;
        std::cout << "\n" << offset_config.api_status(scheme) << std::endl;
    }
    
    // Validate offset configuration
    if (!offset_config.is_valid()) {
        std::cerr << "Error: Invalid offset table configuration" << std::endl;
        return;
    }
    
    // Validate compression for scheme
    if (!is_compression_supported_for_scheme(compression, scheme)) {
        std::cerr << "Error: Compression " << get_compression_name(compression) 
                  << " not supported with " << get_tiling_scheme_name(scheme) 
                  << " scheme" << std::endl;
        return;
    }
    
    // Load input
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* full_image = GDALImageLoader::to_heif_image(gdal_data, output_bit_depth);
    
    int width = gdal_data.width;
    int height = gdal_data.height;
    
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(compression);
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error_simple(error, "get encoder");
    
    if (heif_format != heif_compression_uncompressed && !is_lossless_compression(compression)) {
        heif_encoder_set_lossy_quality(encoder, quality);
    }
    
    // Calculate grid dimensions
    int num_cols = (width + tile_width - 1) / tile_width;
    int num_rows = (height + tile_height - 1) / tile_height;
    
    std::cout << "Creating " << num_cols << "x" << num_rows << " tile grid" << std::endl;
    
    heif_encoding_options* options = heif_encoding_options_alloc();
    options->unci_compression = to_unci_compression(compression);
    
    heif_image_handle* tiled_handle = nullptr;
    heif_image* working_image = full_image;
    bool image_was_padded = false;
    
    // Check if padding is needed for this scheme
    if (needs_padding_for_scheme(scheme, width, height, tile_width, tile_height)) {
        std::cout << "  Image dimensions not aligned with tile size, padding..." << std::endl;
        working_image = pad_image_for_tiling(full_image, tile_width, tile_height, padding_value);
        image_was_padded = true;
        
        // Update dimensions to padded size
        width = heif_image_get_primary_width(working_image);
        height = heif_image_get_primary_height(working_image);
        num_cols = width / tile_width;
        num_rows = height / tile_height;
    }
    
    // Calculate expected table size
    size_t entry_size = offset_config.table_entry_size_bytes();
    size_t total_tiles = num_cols * num_rows;
    size_t table_size = entry_size * total_tiles;
    
    std::cout << "  Offset table: " << entry_size << " bytes/tile × " 
              << total_tiles << " tiles = " << table_size << " bytes total" << std::endl;
    
    // Create tiled image based on scheme
    switch (scheme) {
        case TilingScheme::Grid: {
            // Note: Grid scheme in libheif may not directly support custom offset tables
            // But we can document the configuration for reference
            std::cout << "  Grid scheme - offset config for reference: " 
                      << offset_config.description() << std::endl;
            
            error = heif_context_add_grid_image(ctx, width, height,
                                               num_cols, num_rows,
                                               options, &tiled_handle);
            check_error_simple(error, "add grid image");
            break;
        }
        
#ifdef WITH_UNCOMPRESSED_CODEC
        case TilingScheme::UNCI: {
            heif_unci_image_parameters params;
            memset(&params, 0, sizeof(params));
            params.version = 1;
            params.image_width = width;
            params.image_height = height;
            params.tile_width = tile_width;
            params.tile_height = tile_height;
            params.compression = to_unci_compression(compression);
            
            // Apply offset table configuration to UNCI
            // Note: Check libheif documentation for exact field names
            // These may need to be adapted based on the actual UNCI API

            std::cout << "\n  UNCI Parameters (Current API v1):" << std::endl;
            std::cout << "    ✓ image_width: " << params.image_width << std::endl;
            std::cout << "    ✓ image_height: " << params.image_height << std::endl;
            std::cout << "    ✓ tile_width: " << params.tile_width << std::endl;
            std::cout << "    ✓ tile_height: " << params.tile_height << std::endl;
            std::cout << "    ✓ compression: " << static_cast<int>(params.compression) << std::endl;
            
            std::cout << "\n    ⚠️  NOT AVAILABLE in API (yet):" << std::endl;
            std::cout << "    ❌ offset_field_length" << std::endl;
            std::cout << "    ❌ size_field_length" << std::endl;
            std::cout << "    ❌ interleave_type (TODO in libheif)" << std::endl;
            std::cout << "    ❌ padding (TODO in libheif)" << std::endl;
            
            std::cout << "\n    📝 Your requested config (for documentation):" << std::endl;

            std::cout << "  UNCI Configuration:" << std::endl;
            std::cout << "    Offset field: " << static_cast<int>(offset_config.offset_field_length) << " bits" << std::endl;
            std::cout << "    Size field: " << static_cast<int>(offset_config.size_field_length) << " bits" << std::endl;
            std::cout << "    Sequential: " << (offset_config.tiles_are_sequential ? "yes" : "no") << std::endl;
            
            // Create prototype tile
            heif_image* prototype_tile = nullptr;
            error = heif_image_extract_area(working_image, 0, 0, tile_width, tile_height,
                                           heif_get_global_security_limits(),
                                           &prototype_tile);
            check_error_simple(error, "extract prototype tile");
            
            error = heif_context_add_empty_unci_image(ctx, &params, options,
                                                     prototype_tile, &tiled_handle);
            heif_image_release(prototype_tile);
            check_error_simple(error, "add UNCI image");
            break;
        }
#endif
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        case TilingScheme::TILI: {
            heif_tiled_image_parameters tiled_params;
            memset(&tiled_params, 0, sizeof(tiled_params));
            tiled_params.version = 1;
            tiled_params.image_width = width;
            tiled_params.image_height = height;
            tiled_params.tile_width = tile_width;
            tiled_params.tile_height = tile_height;
            
            // Apply offset table configuration to TILI
            tiled_params.offset_field_length = offset_config.offset_field_length;
            tiled_params.size_field_length = offset_config.size_field_length;
            tiled_params.tiles_are_sequential = offset_config.tiles_are_sequential;
            
            std::cout << "  TILI Configuration:" << std::endl;
            std::cout << "    Offset field: " << static_cast<int>(tiled_params.offset_field_length) << " bits" << std::endl;
            std::cout << "    Size field: " << static_cast<int>(tiled_params.size_field_length) << " bits" << std::endl;
            std::cout << "    Sequential: " << (tiled_params.tiles_are_sequential ? "yes" : "no") << std::endl;
            std::cout << "    Expected table size: " << table_size << " bytes" << std::endl;
            
            error = heif_context_add_tiled_image(ctx, &tiled_params, options,
                                                encoder, &tiled_handle);
            check_error_simple(error, "add TILI image");
            break;
        }
#endif
        
        default:
            std::cerr << "Unsupported tiling scheme" << std::endl;
            if (image_was_padded && working_image != full_image) {
                heif_image_release(working_image);
            }
            heif_encoding_options_free(options);
            heif_encoder_release(encoder);
            heif_context_free(ctx);
            heif_image_release(full_image);
            return;
    }
    
    // Encode all tiles
    for (int ty = 0; ty < num_rows; ty++) {
        for (int tx = 0; tx < num_cols; tx++) {
            if ((ty * num_cols + tx) % 100 == 0) {
                std::cout << "\rEncoding tile (" << tx << "," << ty << ")..." << std::flush;
            }
            
            heif_image* tile_image = nullptr;
            error = heif_image_extract_area(working_image,
                                          tx * tile_width, ty * tile_height,
                                          tile_width, tile_height,
                                          heif_get_global_security_limits(),
                                          &tile_image);
            check_error_simple(error, "extract tile");
            
            error = heif_context_add_image_tile(ctx, tiled_handle, tx, ty,
                                               tile_image, encoder);
            check_error_simple(error, "add tile");
            
            heif_image_release(tile_image);
        }
    }
    std::cout << "\nAll tiles encoded." << std::endl;
    
    heif_context_set_primary_image(ctx, tiled_handle);
    
    // padding
    // Write compact padding metadata if image was padded
    if (image_was_padded && working_image != full_image) {
        if (tiling_config.store_padding_metadata) {  // CHECK CONFIG

            std::cout << "\n  Image (single-level) was padded, writing padding metadata..." << std::endl;
            
            auto padding_meta = CompactPaddingMetadata::create(
                gdal_data.width, gdal_data.height,
                tile_width, tile_height,
                padding_value);

            // Write based on configured format
            switch (tiling_config.padding_format) {
                case PaddingMetadataFormat::TPAD_BINARY:
                    write_tpad_metadata(ctx, tiled_handle, padding_meta, 0);
                    break;
                    
                case PaddingMetadataFormat::XMP_TEXT:
                    write_compact_padding_as_simple_xmp(ctx, tiled_handle, padding_meta, 0);
                    break;
                    
                case PaddingMetadataFormat::BOTH:
                    write_tpad_metadata(ctx, tiled_handle, padding_meta, 0);
                    write_compact_padding_as_simple_xmp(ctx, tiled_handle, padding_meta, 0);
                    break;
            }            
        } else {
            std::cout << "\n  Padding metadata storage disabled by configuration" << std::endl;
        }
    } else {
        std::cout << "\n  No padding applied (image already aligned to tile grid)" << std::endl;
    }

    error = heif_context_write_to_file(ctx, output_file);
    check_error_simple(error, "write to file");
    
    std::cout << "✓ Successfully encoded tiled image: " << output_file << std::endl;
    
    if (image_was_padded && working_image != full_image) {
        heif_image_release(working_image);
    }
    heif_image_handle_release(tiled_handle);
    heif_encoding_options_free(options);
    heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(full_image);
}

std::cout << "Tiled encoding with offset configuration defined." << std::endl;

#endif // HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H

## Comprehensive Tiled Encoding Functions with Automatic Limit Adjustment

In [ ]:
#ifndef HEIF_ENCODE_TILED_COMPREHENSIVE_H
#define HEIF_ENCODE_TILED_COMPREHENSIVE_H


// Backward compatible versions
void encode_tiled_grid(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::Grid, compression, quality, output_bit_depth, padding_value, offset_config);
}

#ifdef WITH_UNCOMPRESSED_CODEC
void encode_tiled_unci(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::UNCI, compression, quality, output_bit_depth, padding_value, offset_config);
}
#endif

#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
void encode_tiled_tili(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::TILI, compression, quality, output_bit_depth, padding_value, offset_config);
}
#endif

std::cout << "Comprehensive tiled encoding functions defined." << std::endl;

#endif // HEIF_ENCODE_TILED_COMPREHENSIVE_H

## Pyramid/Overview Generation with Configurable Levels and padding (if needed)

### Configure and Helper for Write-Order

In [ ]:
#ifndef HEIF_PYRAMID_WRITE_ORDER_OPTIMIZED_H
#define HEIF_PYRAMID_WRITE_ORDER_OPTIMIZED_H

// Pyramid level write order (optional)
enum class PyramidWriteOrder : uint8_t {
    Default,               // Use existing memory-efficient encoding (no reordering)
    HighResolutionFirst,   // Explicit: Level 0 (full res) → Level N (coarsest)
    CoarseFirst           // COG-style: Level N (coarsest) → Level 0 (full res)
};

struct PyramidWriteOrderInfo {
    PyramidWriteOrder order;
    std::string_view name;
    std::string_view description;
};

constexpr inline std::array<PyramidWriteOrderInfo, 3> PYRAMID_WRITE_ORDERS = {{
    {PyramidWriteOrder::Default, "Default", 
     "Memory-efficient: encode and write levels sequentially (high-res first)"},
    {PyramidWriteOrder::HighResolutionFirst, "HighResFirst", 
     "Explicit order: write full resolution first, then overviews"},
    {PyramidWriteOrder::CoarseFirst, "CoarseFirst", 
     "COG-style: write coarsest overview first, then full resolution"}
}};

inline std::string_view get_write_order_name(PyramidWriteOrder order) {
    for (const auto& info : PYRAMID_WRITE_ORDERS) {
        if (info.order == order) {
            return info.name;
        }
    }
    return "Unknown";
}

void list_pyramid_write_orders() {
    std::cout << "\n=== Pyramid Write Order Options ===\n" << std::endl;
    std::cout << std::left << std::setw(20) << "Order" 
              << "Description" << std::endl;
    std::cout << std::string(80, '-') << std::endl;
    
    for (const auto& info : PYRAMID_WRITE_ORDERS) {
        std::cout << std::left << std::setw(20) << info.name
                  << info.description << std::endl;
    }
    
    std::cout << "\n💡 Recommendations:" << std::endl;
    std::cout << "   • Default: Best for memory efficiency (only 2 levels in memory)" << std::endl;
    std::cout << "   • CoarseFirst: Best for web/streaming (COG-style layout)" << std::endl;
    std::cout << "   • HighResFirst: Same result as Default but validates ordering" << std::endl;
    
    std::cout << "\n📝 Note: Primary image always points to full resolution (Level 0)" << std::endl;
    std::cout << "         regardless of write order\n" << std::endl;
}

#endif // HEIF_PYRAMID_WRITE_ORDER_OPTIMIZED_H

### Configure and Help functions

In [ ]:
#ifndef HEIF_ENCODE_PYRAMID_CONFIG_AND_HELPERS_H
#define HEIF_ENCODE_PYRAMID_CONFIG_AND_HELPERS_H

// Structure to define pyramid configuration
struct PyramidConfig {
    int num_levels;                      // Total number of pyramid levels (including full resolution)
    bool use_tiling;                     // Whether to tile each level
    TilingScheme tiling_scheme;          // Tiling scheme to use (if use_tiling is true)
    int tile_width;                      // Tile width (if use_tiling is true)
    int tile_height;                     // Tile height (if use_tiling is true)
    ExtendedCompressionType compression; // Compression for all levels
    int quality;                         // Quality for lossy compression
    int output_bit_depth;                // Bit depth
    uint16_t padding_value;              // Padding value for tiles (if tiling is used)
    OffsetTableConfig offset_config;     // Offset table configuration for tiled levels (if tiling is used)

    // Optional: different settings per level
    std::vector<int> quality_per_level;  // If not empty, use different quality per level

    // NEW: Control padding metadata storage
    bool store_padding_metadata;  // Default: true
    PaddingMetadataFormat padding_format;

    // NEW: Write order control
    PyramidWriteOrder write_order;       // Default: PyramidWriteOrder::Default
        
    //constructor with all parameterers initialized to defaults
    PyramidConfig()
        : num_levels(1)                                          // Default: single level
        , use_tiling(false)                                      // Default: no tiling
        , tiling_scheme(TilingScheme::Grid)                      // Default scheme
        , tile_width(256)                                        // Default tile size
        , tile_height(256)                                       // Default tile size
        , compression(ExtendedCompressionType::HEVC)             // Default compression
        , quality(50)                                            // Default quality
        , output_bit_depth(8)                                    // Default 8-bit
        , padding_value(0)                                       // Default: black padding
        , offset_config(OffsetTableConfig::Preset::Default)      // Default offset config
        , quality_per_level()                                    // Empty vector (use default quality)
        , store_padding_metadata(true)                           // Default: store metadata
        , padding_format(PaddingMetadataFormat::TPAD_BINARY)     // Default: binary format
        , write_order(PyramidWriteOrder::Default)                // Default: memory-efficient
        {}
};

// Helper to downsample image
inline heif_image* downsample_image(heif_image* src, int factor) {
    int src_width = heif_image_get_primary_width(src);
    int src_height = heif_image_get_primary_height(src);
    int dst_width = src_width / factor;
    int dst_height = src_height / factor;
    
    heif_image* dst = nullptr;
    heif_error error = heif_image_create(dst_width, dst_height,
                                        heif_image_get_colorspace(src),
                                        heif_image_get_chroma_format(src),
                                        &dst);
    check_error_simple(error, "create downsampled image");
    
    // Simple nearest-neighbor downsampling
    heif_channel channels[] = {heif_channel_R, heif_channel_G, heif_channel_B, 
                               heif_channel_Y, heif_channel_Alpha};
    for (auto channel : channels) {
        if (!heif_image_has_channel(src, channel)) continue;
        
        int bits = heif_image_get_bits_per_pixel(src, channel);
        heif_image_add_plane(dst, channel, dst_width, dst_height, bits);
        
        int src_stride, dst_stride;
        const uint8_t* src_data = heif_image_get_plane_readonly(src, channel, &src_stride);
        uint8_t* dst_data = heif_image_get_plane(dst, channel, &dst_stride);
        
        for (int y = 0; y < dst_height; y++) {
            for (int x = 0; x < dst_width; x++) {
                dst_data[y * dst_stride + x] = src_data[(y * factor) * src_stride + (x * factor)];
            }
        }
    }
    
    return dst;
}

#endif // HEIF_ENCODE_PYRAMID_CONFIG_AND_HELPERS_H

### Main encode pyramid logic

In [ ]:
#ifndef HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H
#define HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H

// ============================================================================
// Rename existing function to default (memory-efficient) implementation
// ============================================================================

void encode_pyramid_default(const char* input_file, 
                            const char* output_file,
                            const PyramidConfig& config) {
    
    std::cout << "\n=== Encoding Pyramid Image ===" << std::endl;
    std::cout << "Levels: " << config.num_levels << std::endl;
    std::cout << "Compression: " << get_compression_name(config.compression) << std::endl;
    if (config.use_tiling) {
        std::cout << "Tiling: " << get_tiling_scheme_name(config.tiling_scheme) 
                << " (" << config.tile_width << "x" << config.tile_height << ")" << std::endl;
    } else {
        std::cout << "Tiling: None" << std::endl;
    }
    
    std::cout << "Padding Metadata: " << (config.store_padding_metadata ? "Enabled" : "Disabled") << std::endl;
    if (config.store_padding_metadata) {
        switch (config.padding_format) {
            case PaddingMetadataFormat::TPAD_BINARY:
                std::cout << "  Format: TPAD Binary (24 bytes per level)" << std::endl;
                break;
            case PaddingMetadataFormat::XMP_TEXT:
                std::cout << "  Format: XMP Text" << std::endl;
                break;
            case PaddingMetadataFormat::BOTH:
                std::cout << "  Format: Both TPAD and XMP" << std::endl;
                break;
        }
    }

    // Load input
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* full_image = GDALImageLoader::to_heif_image(gdal_data, config.output_bit_depth);
    
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(config.compression);
    heif_error error;
    /*
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error_simple(error, "get encoder");
    */

    heif_encoding_options* options = heif_encoding_options_alloc();
    options->unci_compression = to_unci_compression(config.compression);
    
    std::vector<heif_item_id> pyramid_ids;
    heif_image* current_level = full_image;
    
    // Collect padding metadata for all levels
    MultiLevelPaddingMetadata multilevel_meta = MultiLevelPaddingMetadata::create();
    bool any_level_padded = false;

    // Encode each pyramid level
    for (int level = 0; level < config.num_levels; level++) {
        int curr_width = heif_image_get_primary_width(current_level);
        int curr_height = heif_image_get_primary_height(current_level);

        // *** Create fresh encoder for THIS level ***
        heif_encoder* encoder = nullptr;
        error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
        check_error_simple(error, std::format("get encoder for level {}", level));        
        
        int quality = config.quality;
        if (!config.quality_per_level.empty() && level < config.quality_per_level.size()) {
            quality = config.quality_per_level[level];
        }
        
        if (heif_format != heif_compression_uncompressed && !is_lossless_compression(config.compression)) {
            heif_encoder_set_lossy_quality(encoder, quality);
        }
        
        std::cout << "\nLevel " << level << ": " << curr_width << "x" << curr_height;
        if (!is_lossless_compression(config.compression)) {
            std::cout << " (quality " << quality << ")";
        }
        std::cout << std::endl;
        
        heif_image_handle* level_handle = nullptr;
        heif_image* working_image = current_level;  // Image to use for this level
        bool image_was_padded = false;
        
        if (config.use_tiling) {
            // Check if padding is needed for this scheme
            if (needs_padding_for_scheme(config.tiling_scheme, 
                                        curr_width, curr_height,
                                        config.tile_width, config.tile_height)) {
                std::cout << "  Image dimensions not aligned with tile size, padding..." << std::endl;
                working_image = pad_image_for_tiling(current_level, 
                                                    config.tile_width, 
                                                    config.tile_height,
                                                    config.padding_value);
                image_was_padded = true;
                
                // Update dimensions to padded size
                curr_width = heif_image_get_primary_width(working_image);
                curr_height = heif_image_get_primary_height(working_image);
            }
            
            int num_cols = curr_width / config.tile_width;
            int num_rows = curr_height / config.tile_height;
            
            std::cout << "  Creating " << num_cols << "x" << num_rows << " tile grid" << std::endl;
            
            // Create tiled image structure
            switch (config.tiling_scheme) {
                case TilingScheme::Grid:
                    // Grid can handle original dimensions
                    num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                    num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                    
                    error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                    num_cols, num_rows,
                                                    options, &level_handle);
                    check_error_simple(error, "add grid image");
                    break;
            
#ifdef WITH_UNCOMPRESSED_CODEC
                case TilingScheme::UNCI: {
                    heif_unci_image_parameters params;
                    memset(&params, 0, sizeof(params));
                    params.version = 1;
                    params.image_width = curr_width;
                    params.image_height = curr_height;
                    params.tile_width = config.tile_width;
                    params.tile_height = config.tile_height;
                    params.compression = to_unci_compression(config.compression);
                    
                    std::cout << "  Creating UNCI image: " << params.image_width 
                            << "x" << params.image_height 
                            << " with " << params.tile_width << "x" << params.tile_height 
                            << " tiles" << std::endl;
                    
                    heif_image* prototype_tile = nullptr;
                    error = heif_image_extract_area(working_image, 0, 0, 
                                                config.tile_width, config.tile_height,
                                                nullptr, &prototype_tile);
                    check_error_simple(error, "extract prototype tile");
                    
                    error = heif_context_add_empty_unci_image(ctx, &params, options,
                                                            prototype_tile, &level_handle);
                    heif_image_release(prototype_tile);
                    check_error_simple(error, "add UNCI image");
                    break;
                }
#endif //WITH_UNCOMPRESSED_CODEC

#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
                case TilingScheme::TILI: {
                    heif_tiled_image_parameters tiled_params;
                    memset(&tiled_params, 0, sizeof(tiled_params));
                    tiled_params.version = 1;
                    tiled_params.image_width = curr_width;
                    tiled_params.image_height = curr_height;
                    tiled_params.tile_width = config.tile_width;
                    tiled_params.tile_height = config.tile_height;

                    // Apply offset configuration from config
                    tiled_params.offset_field_length = config.offset_config.offset_field_length;
                    tiled_params.size_field_length = config.offset_config.size_field_length;
                    tiled_params.tiles_are_sequential = config.offset_config.tiles_are_sequential;

                    std::cout << "  Creating TILI structure for level " << level << std::endl;
                    
                    error = heif_context_add_tiled_image(ctx, &tiled_params, options,
                                                        encoder, &level_handle);
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  TILI failed at level " << level << ": " 
                                << error.message << std::endl;
                        std::cout << "  Falling back to Grid for this level" << std::endl;
                        
                        num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                        num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                        
                        error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                        num_cols, num_rows,
                                                        options, &level_handle);
                        check_error_simple(error, "add grid image (TILI fallback)");
                    }
                    break;
                }
#endif //HEIF_ENABLE_EXPERIMENTAL_FEATURES
            }
 
            // Encode tiles for this level -Update Pyramid Encoding to Use Padding for TILI
            std::cout << "  Encoding " << (num_cols * num_rows) << " tiles..." << std::endl;
            
            for (int ty = 0; ty < num_rows; ty++) {
                for (int tx = 0; tx < num_cols; tx++) {
                    int tile_x = tx * config.tile_width;
                    int tile_y = ty * config.tile_height;

                    // *** FIX: Calculate actual tile dimensions ***
                    int actual_tile_width = std::min(config.tile_width, curr_width - tile_x);
                    int actual_tile_height = std::min(config.tile_height, curr_height - tile_y);
                    
                    // Skip if tile is completely outside image bounds
                    if (tile_x >= curr_width || tile_y >= curr_height) {
                        std::cout << "  Skipping out-of-bounds tile (" << tx << "," << ty << ")" << std::endl;
                        continue;
                    }
                    
                    //std::cout << "\rEncoding tile (" << tx << "," << ty << ") at (" 
                    //        << tile_x << "," << tile_y << ") size " 
                    //        << actual_tile_width << "x" << actual_tile_height << "..." << std::flush;
                    
                    // Extract tile - use exact tile size since image is now padded
                    heif_image* tile_image = nullptr;
                    error = heif_image_extract_area(working_image,
                                                tile_x, tile_y,
                                                actual_tile_width, actual_tile_height, //config.tile_width, config.tile_height,
                                                nullptr, &tile_image);
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  Failed to extract tile (" << tx << "," << ty 
                                << ") at level " << level << ": " << error.message << std::endl;
                        
                        // Cleanup
                        if (image_was_padded && working_image != current_level) {
                            heif_image_release(working_image);
                        }
                        if (level_handle) heif_image_handle_release(level_handle);
                        heif_encoding_options_free(options);
                        heif_encoder_release(encoder);
                        heif_context_free(ctx);
                        if (current_level != full_image) heif_image_release(current_level);
                        heif_image_release(full_image);
                        return;
                    }
                    
                    // Add tile
                    error = heif_context_add_image_tile(ctx, level_handle, tx, ty,
                                                    tile_image, encoder);
                    
                    heif_image_release(tile_image);
                    tile_image = nullptr;
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  Failed to add tile (" << tx << "," << ty 
                                << ") at level " << level << ": " << error.message << std::endl;
                        
                        // Cleanup
                        if (image_was_padded && working_image != current_level) {
                            heif_image_release(working_image);
                        }
                        if (level_handle) heif_image_handle_release(level_handle);
                        heif_encoding_options_free(options);
                        heif_encoder_release(encoder);
                        heif_context_free(ctx);
                        if (current_level != full_image) heif_image_release(current_level);
                        heif_image_release(full_image);
                        return;
                    }
                    
                    // Progress
                    if ((ty * num_cols + tx + 1) % 50 == 0) {
                        std::cout << "    Progress: " << (ty * num_cols + tx + 1) 
                                << "/" << (num_cols * num_rows) << std::endl;
                    }
                }
            }
        
  
            std::cout << "  ✓ All tiles encoded for level " << level << std::endl;

            // Release padded image if it was created
            if (image_was_padded && working_image != current_level) {
                heif_image_release(working_image);
                working_image = nullptr;
            }
            
        } else {
            // Non-tiled level
            error = heif_context_encode_image(ctx, current_level, encoder, options, &level_handle);
            check_error_simple(error, "encode level");
        }
        
        // Set first level as primary
        if (level == 0) {
            heif_context_set_primary_image(ctx, level_handle);
        }

        // At end of level, release THIS level's encoder
        heif_encoder_release(encoder);
        encoder = nullptr;

        // Write compact padding metadata for this level if image was padded
        if (image_was_padded && working_image != current_level) {
            if (config.store_padding_metadata) {  // CHECK CONFIG
                //std::cout << "  Writing padding metadata for level " << level << "..." << std::endl;
                
                // Original dimensions for this level
                int level_orig_width = heif_image_get_primary_width(current_level);
                int level_orig_height = heif_image_get_primary_height(current_level);
                
                auto level_padding_meta = CompactPaddingMetadata::create(
                    level_orig_width, level_orig_height,
                    config.tile_width, config.tile_height,
                    config.padding_value);

                multilevel_meta.add_level(level_padding_meta);
                any_level_padded = true;
                
                std::cout << "  Collected padding info for level " << level << std::endl;
            } else {
                std::cout << "  Padding metadata storage disabled for level " << level << std::endl;
            }
        }
        

        pyramid_ids.push_back(heif_image_handle_get_item_id(level_handle));
        heif_image_handle_release(level_handle);
        level_handle = nullptr;
        
        // Prepare next level
        if (level < config.num_levels - 1) {
            std::cout << "  Downsampling for next level..." << std::endl;
            
            heif_image* downsampled = downsample_image(current_level, 2);
            
            // CRITICAL: Release previous level image (except the original full_image)
            if (current_level != full_image) {
                heif_image_release(current_level);
            }
            current_level = downsampled;
            
            std::cout << "  ✓ Next level prepared: " 
                      << heif_image_get_primary_width(current_level) << "x"
                      << heif_image_get_primary_height(current_level) << std::endl;
        }
    }
    
    // Clean up the last level image (if it's not the original)
    if (current_level != full_image) {
        heif_image_release(current_level);
    }
    current_level = nullptr;
    
    // Create pyramid entity group
    std::cout << "\nCreating pyramid entity group..." << std::endl;
    error = heif_context_add_pyramid_entity_group(ctx, pyramid_ids.data(),
                                                  pyramid_ids.size(), nullptr);
    check_error_simple(error, "add pyramid entity group");
    
    // Write multi-level metadata ONCE at the end (to primary image handle)
    if (any_level_padded && config.store_padding_metadata) {
        std::cout << "\nWriting collected padding metadata for all levels..." << std::endl;
        multilevel_meta.print();
        
        // Get primary image handle
        heif_image_handle* primary_handle;
        heif_error error = heif_context_get_primary_image_handle(ctx, &primary_handle);
        if (error.code == heif_error_Ok) {
            write_multilevel_padding_metadata(ctx, primary_handle, multilevel_meta, 
                                             config.padding_format);
            heif_image_handle_release(primary_handle);
        } else {
            std::cerr << "  ⚠ Could not get primary handle to write metadata: " 
                      << error.message << std::endl;
        }
    }


    // Write to file
    std::cout << "Writing to file..." << std::endl;
    error = heif_context_write_to_file(ctx, output_file);
    check_error_simple(error, "write to file");
    
    std::cout << "\n✓ Successfully encoded pyramid with " << config.num_levels 
              << " levels: " << output_file << std::endl;
    
    // Final cleanup
    heif_encoding_options_free(options);
    // heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(full_image);
}


// ============================================================================
// COG-style encoder (coarse first)
// ============================================================================

void encode_pyramid_cog(const char* input_file, 
                       const char* output_file,
                       const PyramidConfig& config) {
    
    std::cout << "\n=== Encoding Pyramid Image (COG-Style: Coarse First) ===" << std::endl;
    std::cout << "Levels: " << config.num_levels << std::endl;
    std::cout << "Compression: " << get_compression_name(config.compression) << std::endl;
    
    if (config.use_tiling) {
        std::cout << "Tiling: " << get_tiling_scheme_name(config.tiling_scheme) 
                  << " (" << config.tile_width << "x" << config.tile_height << ")" << std::endl;
    } else {
        std::cout << "Tiling: None" << std::endl;
    }
    
    std::cout << "Padding Metadata: " << (config.store_padding_metadata ? "Enabled" : "Disabled") << std::endl;
    
    // Load input
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* full_image = GDALImageLoader::to_heif_image(gdal_data, config.output_bit_depth);
    
    heif_context* ctx = heif_context_alloc();
    heif_encoding_options* options = heif_encoding_options_alloc();
    options->unci_compression = to_unci_compression(config.compression);
    heif_compression_format heif_format = to_heif_compression_format(config.compression);
    
    // ========================================================================
    // STEP 1: Generate all pyramid levels
    // ========================================================================
    std::cout << "\n--- Step 1: Generating all pyramid levels ---\n" << std::endl;
    
    std::vector<heif_image*> all_levels;
    heif_image* current_level = full_image;
    
    for (int level = 0; level < config.num_levels; level++) {
        all_levels.push_back(current_level);
        
        int curr_width = heif_image_get_primary_width(current_level);
        int curr_height = heif_image_get_primary_height(current_level);
        std::cout << "Level " << level << ": " << curr_width << "x" << curr_height << std::endl;
        
        // Downsample for next level
        if (level < config.num_levels - 1) {
            heif_image* downsampled = downsample_image(current_level, 2);
            current_level = downsampled;
        }
    }
    
    // ========================================================================
    // STEP 2: Encode in reverse order (coarse first)
    // ========================================================================
    std::cout << "\n--- Step 2: Encoding in COG order (coarse → fine) ---\n" << std::endl;
    
    std::vector<heif_item_id> pyramid_ids(config.num_levels);
    MultiLevelPaddingMetadata multilevel_meta = MultiLevelPaddingMetadata::create();
    
    for (int write_idx = config.num_levels - 1; write_idx >= 0; write_idx--) {
        int level = write_idx;
        heif_image* level_image = all_levels[level];
        
        int curr_width = heif_image_get_primary_width(level_image);
        int curr_height = heif_image_get_primary_height(level_image);
        
        std::cout << "\nEncoding level " << level 
                  << " (write position " << (config.num_levels - 1 - write_idx) << ")" << std::endl;
        
        // Create encoder for this level
        heif_encoder* encoder = nullptr;
        heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
        check_error_simple(error, std::format("get encoder for level {}", level));
        
        int quality = config.quality;
        if (!config.quality_per_level.empty() && level < config.quality_per_level.size()) {
            quality = config.quality_per_level[level];
        }
        
        if (heif_format != heif_compression_uncompressed && 
            !is_lossless_compression(config.compression)) {
            heif_encoder_set_lossy_quality(encoder, quality);
        }
        
        std::cout << "Level " << level << ": " << curr_width << "x" << curr_height;
        if (!is_lossless_compression(config.compression)) {
            std::cout << " (quality " << quality << ")";
        }
        std::cout << std::endl;
        
        heif_image_handle* level_handle = nullptr;
        heif_image* working_image = level_image;
        bool image_was_padded = false;
        
        if (config.use_tiling) {
            // Check if padding is needed
            if (needs_padding_for_scheme(config.tiling_scheme, 
                                        curr_width, curr_height,
                                        config.tile_width, config.tile_height)) {
                std::cout << "  Padding required..." << std::endl;
                working_image = pad_image_for_tiling(level_image, 
                                                    config.tile_width, 
                                                    config.tile_height,
                                                    config.padding_value);
                image_was_padded = true;
                
                curr_width = heif_image_get_primary_width(working_image);
                curr_height = heif_image_get_primary_height(working_image);
            }
            
            int num_cols = curr_width / config.tile_width;
            int num_rows = curr_height / config.tile_height;
            
            std::cout << "  Creating " << num_cols << "x" << num_rows << " tile grid" << std::endl;
            
            // Create tiled image structure based on scheme
            switch (config.tiling_scheme) {
                case TilingScheme::Grid:
                    num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                    num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                    
                    error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                       num_cols, num_rows,
                                                       options, &level_handle);
                    check_error_simple(error, "add grid image");
                    break;
                
#ifdef WITH_UNCOMPRESSED_CODEC
                case TilingScheme::UNCI: {
                    heif_unci_image_parameters params;
                    memset(&params, 0, sizeof(params));
                    params.version = 1;
                    params.image_width = curr_width;
                    params.image_height = curr_height;
                    params.tile_width = config.tile_width;
                    params.tile_height = config.tile_height;
                    params.compression = to_unci_compression(config.compression);
                    
                    heif_image* prototype_tile = nullptr;
                    error = heif_image_extract_area(working_image, 0, 0, 
                                                   config.tile_width, config.tile_height,
                                                   nullptr, &prototype_tile);
                    check_error_simple(error, "extract prototype tile");
                    
                    error = heif_context_add_empty_unci_image(ctx, &params, options,
                                                             prototype_tile, &level_handle);
                    heif_image_release(prototype_tile);
                    check_error_simple(error, "add UNCI image");
                    break;
                }
#endif //WITH_UNCOMPRESSED_CODEC

#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES                
                case TilingScheme::TILI: {
                    heif_tiled_image_parameters tiled_params;
                    memset(&tiled_params, 0, sizeof(tiled_params));
                    tiled_params.version = 1;
                    tiled_params.image_width = curr_width;
                    tiled_params.image_height = curr_height;
                    tiled_params.tile_width = config.tile_width;
                    tiled_params.tile_height = config.tile_height;
                    tiled_params.offset_field_length = config.offset_config.offset_field_length;
                    tiled_params.size_field_length = config.offset_config.size_field_length;
                    tiled_params.tiles_are_sequential = config.offset_config.tiles_are_sequential;
                    
                    error = heif_context_add_tiled_image(ctx, &tiled_params, options,
                                                        encoder, &level_handle);
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  TILI failed, falling back to Grid" << std::endl;
                        num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                        num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                        error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                           num_cols, num_rows,
                                                           options, &level_handle);
                        check_error_simple(error, "add grid image (TILI fallback)");
                    }
                    break;
                }
#endif //HEIF_ENABLE_EXPERIMENTAL_FEATURES
            }
            
            // Encode tiles
            std::cout << "  Encoding " << (num_cols * num_rows) << " tiles..." << std::endl;
            for (int ty = 0; ty < num_rows; ty++) {
                for (int tx = 0; tx < num_cols; tx++) {
                    int tile_x = tx * config.tile_width;
                    int tile_y = ty * config.tile_height;
                    int actual_tile_width = std::min(config.tile_width, curr_width - tile_x);
                    int actual_tile_height = std::min(config.tile_height, curr_height - tile_y);
                    
                    if (tile_x >= curr_width || tile_y >= curr_height) continue;
                    
                    heif_image* tile_image = nullptr;
                    error = heif_image_extract_area(working_image, tile_x, tile_y,
                                                   actual_tile_width, actual_tile_height,
                                                   nullptr, &tile_image);
                    check_error_simple(error, "extract tile");
                    
                    error = heif_context_add_image_tile(ctx, level_handle, tx, ty,
                                                       tile_image, encoder);
                    heif_image_release(tile_image);
                    check_error_simple(error, "add tile");
                    
                    if ((ty * num_cols + tx + 1) % 50 == 0) {
                        std::cout << "    Progress: " << (ty * num_cols + tx + 1) 
                                  << "/" << (num_cols * num_rows) << std::endl;
                    }
                }
            }
            std::cout << "  ✓ All tiles encoded" << std::endl;
            
            if (image_was_padded && working_image != level_image) {
                heif_image_release(working_image);
            }
        } else {
            // Non-tiled
            error = heif_context_encode_image(ctx, level_image, encoder, options, &level_handle);
            check_error_simple(error, "encode level");
        }
        
        // Set level 0 as primary
        if (level == 0) {
            heif_context_set_primary_image(ctx, level_handle);
        }
        
        pyramid_ids[level] = heif_image_handle_get_item_id(level_handle);
        heif_image_handle_release(level_handle);
        heif_encoder_release(encoder);
    }
    
    // ========================================================================
    // STEP 3: Collect padding metadata (in correct level order)
    // ========================================================================
    if (config.store_padding_metadata) {
        for (int level = 0; level < config.num_levels; level++) {
            int curr_width = heif_image_get_primary_width(all_levels[level]);
            int curr_height = heif_image_get_primary_height(all_levels[level]);
            
            if (config.use_tiling && 
                needs_padding_for_scheme(config.tiling_scheme, 
                                        curr_width, curr_height,
                                        config.tile_width, config.tile_height)) {
                
                auto padding_meta = CompactPaddingMetadata::create(
                    curr_width, curr_height,
                    config.tile_width, config.tile_height,
                    config.padding_value);
                multilevel_meta.add_level(padding_meta);
            }
        }
    }
    
    // ========================================================================
    // STEP 4: Cleanup all levels
    // ========================================================================
    for (size_t i = 0; i < all_levels.size(); i++) {
        if (all_levels[i] != full_image) {
            heif_image_release(all_levels[i]);
        }
    }
    
    // ========================================================================
    // STEP 5: Finalize
    // ========================================================================
    std::cout << "\nCreating pyramid entity group..." << std::endl;
    heif_error error = heif_context_add_pyramid_entity_group(ctx, pyramid_ids.data(),
                                                             pyramid_ids.size(), nullptr);
    check_error_simple(error, "add pyramid entity group");
    
    // Write metadata
    if (multilevel_meta.num_levels > 0 && config.store_padding_metadata) {
        std::cout << "\nWriting padding metadata..." << std::endl;
        multilevel_meta.print();
        
        heif_image_handle* primary_handle;
        error = heif_context_get_primary_image_handle(ctx, &primary_handle);
        if (error.code == heif_error_Ok) {
            write_multilevel_padding_metadata(ctx, primary_handle, multilevel_meta, 
                                             config.padding_format);
            heif_image_handle_release(primary_handle);
        }
    }
    
    // Write file
    std::cout << "\nWriting to file..." << std::endl;
    error = heif_context_write_to_file(ctx, output_file);
    check_error_simple(error, "write to file");
    
    std::cout << "\n✓ Successfully encoded COG-style pyramid with " << config.num_levels 
              << " levels: " << output_file << std::endl;
    
    heif_encoding_options_free(options);
    heif_context_free(ctx);
    heif_image_release(full_image);
}

// ============================================================================
// Main dispatcher function
// ============================================================================


void encode_pyramid_with_config(const char* input_file, 
                               const char* output_file,
                               const PyramidConfig& config) {
    
    // Route to appropriate implementation based on write order
    switch (config.write_order) {
        case PyramidWriteOrder::Default:
        case PyramidWriteOrder::HighResolutionFirst:
            // Use memory-efficient default encoder
            encode_pyramid_default(input_file, output_file, config);
            break;
            
        case PyramidWriteOrder::CoarseFirst:
            // Use COG-style encoder
            encode_pyramid_cog(input_file, output_file, config);
            break;
            
        default:
            std::cerr << "Unknown write order: " 
                      << static_cast<int>(config.write_order) << std::endl;
            std::cerr << "Falling back to default encoder" << std::endl;
            encode_pyramid_default(input_file, output_file, config);
            break;
    }
}

std::cout << "Pyramid encoding with configurable write order defined." << std::endl;

#endif // HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H

### Compatability Wrappers

In [ ]:
#ifndef HEIF_ENCODE_PYRAMID_COMPATABILITY_H
#define HEIF_ENCODE_PYRAMID_COMPATABILITY_H

// Simplified pyramid encoding (backward compatible)
void encode_pyramid(const char* input_file, 
                   const char* output_file,
                   int num_levels,
                   ExtendedCompressionType compression,
                   int quality = 50,
                   int output_bit_depth = 8) {
    PyramidConfig config;
    // memset(&config, 0, sizeof(config)); // cannot be used for complex structs with non-trivial members
    config.num_levels = num_levels;
    config.use_tiling = false;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    
    encode_pyramid_with_config(input_file, output_file, config);
}



// Tiled pyramid encoding
void encode_tiled_pyramid(const char* input_file, 
                         const char* output_file,
                         int num_levels,
                         int tile_width,
                         int tile_height,
                         TilingScheme scheme,
                         ExtendedCompressionType compression,
                         int quality = 50,
                         int output_bit_depth = 8,
                         uint16_t padding_value = 0,
                         const OffsetTableConfig& offset_config = OffsetTableConfig(),
                         const TilingConfig& tiling_config = TilingConfig(),
                         const PyramidWriteOrder& write_order_config = PyramidWriteOrder::Default) {
    PyramidConfig config;
    // memset(&config, 0, sizeof(config)); // cannot be used for complex structs with non-trivial members
    config.num_levels = num_levels;
    config.use_tiling = true;
    config.tiling_scheme = scheme;
    config.tile_width = tile_width;
    config.tile_height = tile_height;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    config.padding_value = padding_value;
    config.offset_config = offset_config;
    config.store_padding_metadata = tiling_config.store_padding_metadata;
    config.padding_format = tiling_config.padding_format;
    config.write_order = write_order_config;
    
    encode_pyramid_with_config(input_file, output_file, config);
}


#endif // HEIF_ENCODE_PYRAMID_COMPATABILITY_H

### Encode tiled pyramdi with config plus padding

## Test All Tiling Schemes and Pyramid Options

In [ ]:
#ifndef HEIF_TEST_TILING_H
#define HEIF_TEST_TILING_H

void test_all_tiling_schemes(const char* input_file, const char* output_dir = ".") {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "Testing All Tiling Schemes" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    std::string outputFileStr = (fs::path(output_dir) / "test_all_tiling_schemes_output.txt").string();
    const char* output_file = outputFileStr.c_str();

    try {
        // Test Grid tiling with different compressions
        std::cout << "\n1. Grid Tiling Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_grid_512_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_grid_512_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_grid_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::DEFLATE);
        
#ifdef WITH_UNCOMPRESSED_CODEC
        // Test UNCI tiling
        std::cout << "\n2. UNCI Tiling Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_unci_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::UNCI, ExtendedCompressionType::DEFLATE);
        outputFileStr = (fs::path(output_dir) / "output_unci_512_zlib.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::UNCI, ExtendedCompressionType::ZLIB);
#endif
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        // Test TILI tiling
        std::cout << "\n3. TILI Tiling Tests:" << std::endl;
        outputFileStr = fs::path(output_dir) / "output_tili_512_hevc_70.heif";
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::HEVC, 70);

        // Note: AV1 with TILI may not be supported in all implementations. Encoder reported errors.
        // outputFileStr = fs::path(output_dir) / "output_tili_512_av1_60.avif";
        // output_file = outputFileStr.c_str();
        // encode_tiled(input_file, output_file, 512, 512,
        //            TilingScheme::TILI, ExtendedCompressionType::AV1, 60);

        // Test with DEFLATE (should work)
        std::cout << "\n   Testing TILI with DEFLATE (lossless)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tili_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::DEFLATE);
        
        // Test with uncompressed (should work)
        std::cout << "\n   Testing TILI with uncompressed..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tili_512_uncompressed.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::Uncompressed);
        
        // Test pyramids
        std::cout << "\n4. Pyramid Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_pyramid_3levels_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 3,
                      ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_pyramid_5levels_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 5,
                      ExtendedCompressionType::AV1, 60);
        
        // Test tiled pyramids
        std::cout << "\n5. Tiled Pyramid Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tiled_pyramid_4levels_grid_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_pyramid(input_file, output_file,
                           4, 512, 512, TilingScheme::Grid,
                           ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_tiled_pyramid_4levels_tili_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_pyramid(input_file, output_file,
                           4, 512, 512, TilingScheme::TILI,
                           ExtendedCompressionType::DEFLATE);
#endif
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "✓ All tiling scheme tests completed!" << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during testing: " << e.what() << std::endl;
    }
}

#endif // HEIF_TEST_TILING_H

std::cout << "\nTest functions defined." << std::endl;
std::cout << "To test all schemes, call: test_all_tiling_schemes(\"input.tif\", \"output_dir\");" << std::endl;


## Demo and Example Functions (Fixed with proper namespace)

In [ ]:
#ifndef HEIF_DEMO_FUNCTIONS_H
#define HEIF_DEMO_FUNCTIONS_H

void demo_all_compressions(const char* input_file, const char* output_dir) {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "HEIF Encoding Demo - All Compression Methods" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    std::string outputFileStr= (fs::path(output_dir) / "output_all_compressions.heif").string();
    char const* output_file = outputFileStr.c_str();
    
    try {
        // Non-tiled, various compressions
        outputFileStr = (fs::path(output_dir) / "output_nontiled_none.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::None);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_hevc_q50.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 50);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_hevc_q80.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 80);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_av1.avif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_zlib.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::ZLIB);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_brotli.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::Brotli);
        
        // Tiled with different compressions
        outputFileStr = (fs::path(output_dir) / "output_tiled_hevc.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::HEVC, 60);
        outputFileStr = (fs::path(output_dir) / "output_tiled_av1.avif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_tiled_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::DEFLATE);
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        // Pyramid
        outputFileStr = (fs::path(output_dir) / "output_pyramid_hevc.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 4, ExtendedCompressionType::HEVC, 60);
        outputFileStr = (fs::path(output_dir) / "output_pyramid_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 4, ExtendedCompressionType::DEFLATE);
#endif
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "All encodings completed successfully!" << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during encoding: " << e.what() << std::endl;
    }
}

// Simplified demo with just a few formats
void demo_basic_encoding(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Basic Encoding Demo ===" << std::endl;
    std::string outputFileStr = (fs::path(output_dir) / "output_basic.heif").string();
    const char* output_file = outputFileStr.c_str();

    try {
        // Test a few key formats
        std::cout << "\n1. Encoding with HEVC (quality 70)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 70);
        
        std::cout << "\n2. Encoding with AV1 (quality 60)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::AV1, 60);
        
        std::cout << "\n3. Encoding with DEFLATE (lossless)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);
        
        std::cout << "\n4. Encoding tiled image with HEVC..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tiled_512_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::HEVC, 70);
        
        std::cout << "\n✓ Basic encoding demo completed successfully!" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during encoding: " << e.what() << std::endl;
    }
}

// Example: Encode with specific compression type
void encode_example(const char* input_file, 
                   const char* output_file,
                   ExtendedCompressionType compression = ExtendedCompressionType::HEVC,
                   int quality = 70) {
    // Use the helper function to get compression info
    auto compression_name = get_compression_name(compression);
    bool lossless = is_lossless_compression(compression);
    
    std::cout << "\n=== Encoding Example ===" << std::endl;
    std::cout << "Input: " << input_file << std::endl;
    std::cout << "Output: " << output_file << std::endl;
    std::cout << "Compression: " << compression_name << std::endl;
    if (!lossless) {
        std::cout << "Quality: " << quality << std::endl;
    } else {
        std::cout << "Mode: Lossless" << std::endl;
    }
    
    try {
        encode_nontiled(input_file, output_file, compression, quality);
        std::cout << "✓ Encoding completed successfully!" << std::endl;
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
    }
}

// Print compression info for a specific type
void print_compression_info(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    if (!info) {
        std::cout << "Unknown compression type" << std::endl;
        return;
    }
    
    std::cout << "\nCompression Info:" << std::endl;
    std::cout << "  Name: " << info->name << std::endl;
    std::cout << "  Description: " << info->description << std::endl;
    std::cout << "  Lossless: " << (info->lossless ? "Yes" : "No") << std::endl;
    std::cout << "  Supported by libheif: " << (info->supported_by_libheif ? "Yes" : "No") << std::endl;
    std::cout << "  Supported by GDAL: " << (info->supported_by_gdal ? "Yes" : "No") << std::endl;
}

std::cout << "Demo functions defined." << std::endl;
std::cout << "\nAvailable demo functions:" << std::endl;
std::cout << "  - demo_all_compressions(input_file)" << std::endl;
std::cout << "  - demo_basic_encoding(input_file)" << std::endl;
std::cout << "  - encode_example(input_file, output_file, compression, quality)" << std::endl;
std::cout << "  - print_compression_info(compression_type)" << std::endl;

#endif // HEIF_DEMO_FUNCTIONS_H

## Usage Examples

In [ ]:
#ifndef HEIF_USAGE_EXAMPLES_H
#define HEIF_USAGE_EXAMPLES_H

void show_usage_examples() {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "HEIF Encoder Usage Examples" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    
    std::cout << "1. Basic encoding with HEVC:" << std::endl;
    std::cout << "   encode_example(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                  ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "2. Encoding with AV1 (AVIF):" << std::endl;
    std::cout << "   encode_example(\"input.tif\", \"output.avif\", " << std::endl;
    std::cout << "                  ExtendedCompressionType::AV1, 60);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "3. Lossless encoding with DEFLATE:" << std::endl;
    std::cout << "   encode_nontiled(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                   ExtendedCompressionType::DEFLATE);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "4. Tiled encoding:" << std::endl;
    std::cout << "   encode_tiled_grid(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                     512, 512,  // tile size" << std::endl;
    std::cout << "                     ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
    
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
    std::cout << "5. Pyramid/multi-resolution encoding:" << std::endl;
    std::cout << "   encode_pyramid(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                  4,  // number of levels" << std::endl;
    std::cout << "                  ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
#endif
    
    std::cout << "6. Run all compression demos:" << std::endl;
    std::cout << "   demo_all_compressions(\"input.tif\", \"output_dir\");" << std::endl;
    std::cout << std::endl;
    
    std::cout << "7. Run basic demo:" << std::endl;
    std::cout << "   demo_basic_encoding(\"input.tif\", \"output_dir\");" << std::endl;
    std::cout << std::endl;
    
    std::cout << "Available compression types:" << std::endl;
    std::cout << "  - ExtendedCompressionType::None" << std::endl;
    std::cout << "  - ExtendedCompressionType::HEVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::AV1" << std::endl;
    std::cout << "  - ExtendedCompressionType::VVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::AVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::JPEG" << std::endl;
    std::cout << "  - ExtendedCompressionType::JPEG2000" << std::endl;
    std::cout << "  - ExtendedCompressionType::HTJ2K" << std::endl;
    std::cout << "  - ExtendedCompressionType::DEFLATE" << std::endl;
    std::cout << "  - ExtendedCompressionType::ZLIB" << std::endl;
    std::cout << "  - ExtendedCompressionType::Brotli" << std::endl;
    std::cout << "  - ExtendedCompressionType::ZSTD" << std::endl;
    std::cout << "  - ExtendedCompressionType::LZW" << std::endl;
    std::cout << "  - ExtendedCompressionType::WebP" << std::endl;
    std::cout << "  - ExtendedCompressionType::LERC" << std::endl;
    std::cout << "  - ExtendedCompressionType::PackBits" << std::endl;
    
    std::cout << "\n" << std::string(80, '=') << std::endl;
}

show_usage_examples();

#endif // HEIF_USAGE_EXAMPLES_H

In [ ]:
// // Use default padding (0)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE);

// // Specify custom padding value (e.g., 255 for white)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 255);

// // Pyramid with custom padding
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 8, 128);  // Pad with mid-gray (128)
// 8-bit data with default padding (0)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0);

// // 8-bit data with white padding (255)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 255);

// // 10-bit data with max value padding (1023 = 2^10 - 1)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 10, 1023);

// // 12-bit data with mid-gray padding (2048 = 2^11)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 12, 2048);

// // 16-bit data with custom padding (32768)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 16, 32768);

// // Pyramid with 12-bit data and black padding
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 12, 0);


// // Example 1: Default offset configuration (offset + size)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE);

// // Example 2: Using preset - offset and size tables
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config1);

// // Example 3: Using preset - offset only
// OffsetTableConfig config2(OffsetTableConfig::Preset::OffsetOnly);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config2);

// // Example 4: Using preset - size only
// OffsetTableConfig config3(OffsetTableConfig::Preset::SizeOnly);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config3);

// // Example 5: Using preset - sequential (no offset table)
// OffsetTableConfig config4(OffsetTableConfig::Preset::Sequential);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config4);

// // Example 6: Custom offset configuration - 40-bit offset, 32-bit size
// OffsetTableConfig config5(40, 32, false);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config5);

// // Example 7: Custom for 64-bit offsets (large files)
// OffsetTableConfig config6(64, 32, false);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config6);

// // Example 8: Pyramid with custom offset configuration
// OffsetTableConfig pyramid_config(OffsetTableConfig::Preset::OffsetAndSize);
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 8, 0, pyramid_config);

// // Example 9: Compare different configurations
// list_offset_table_presets();

// // Example 10: Test all three schemes with same offset config
// OffsetTableConfig test_config(32, 24, false);

// encode_tiled(input_file, "output_grid.heif", 256, 256,
//             TilingScheme::Grid, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

// encode_tiled(input_file, "output_unci.heif", 256, 256,
//             TilingScheme::UNCI, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

// encode_tiled(input_file, "output_tili.heif", 256, 256,
//             TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

## Helper Function to show offset configuration info

In [ ]:
#ifndef HELPER_FUNCTION_OFFSET_TABLES_H
#define HELPER_FUNCTION_OFFSET_TABLES_H
void demonstrate_offset_configurations(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Demonstrating Offset Table Configurations ===" << std::endl;
    
    list_offset_table_presets_with_support();
    
    std::cout << "\n--- Testing Different Configurations ---" << std::endl;
    
    // Test 1: Offset + Size (standard)
    std::cout << "\n1. Testing Offset + Size configuration:" << std::endl;
    OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
    std::string output1 = std::string(output_dir) + "/tili_offset_and_size.heif";
    encode_tiled_tili(input_file, output1.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config1);
    
    // Test 2: Offset only
    std::cout << "\n2. Testing Offset-only configuration:" << std::endl;
    OffsetTableConfig config2(OffsetTableConfig::Preset::OffsetOnly);
    std::string output2 = std::string(output_dir) + "/tili_offset_only.heif";
    encode_tiled_tili(input_file, output2.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config2);
    
    // Test 3: Size only
    std::cout << "\n3. Testing Size-only configuration:" << std::endl;
    OffsetTableConfig config3(OffsetTableConfig::Preset::SizeOnly);
    std::string output3 = std::string(output_dir) + "/tili_size_only.heif";
    encode_tiled_tili(input_file, output3.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config3);
    
    // Test 4: Sequential
    std::cout << "\n4. Testing Sequential configuration (no offset table):" << std::endl;
    OffsetTableConfig config4(OffsetTableConfig::Preset::Sequential);
    std::string output4 = std::string(output_dir) + "/tili_sequential.heif";
    encode_tiled_tili(input_file, output4.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config4);
    
    std::cout << "\n✓ All offset configuration tests completed!" << std::endl;
    std::cout << "\nCheck output files to compare file sizes and structures." << std::endl;
}
#endif // HELPER_FUNCTION_OFFSET_TABLES_H

In [ ]:
// list_offset_table_presets();
// demonstrate_offset_configurations("input.tif", "output_dir");

## Test Helper Functions

In [ ]:
#ifndef HEIF_TEST_HELPERS_H
#define HEIF_TEST_HELPERS_H

// Test that helper functions work correctly
void test_helper_functions() {
    std::cout << "\n=== Testing Helper Functions ===" << std::endl;
    
    // Test get_compression_info
    std::cout << "\n1. Testing get_compression_info():" << std::endl;
    print_compression_info(ExtendedCompressionType::HEVC);
    print_compression_info(ExtendedCompressionType::DEFLATE);
    
    // Test get_compression_name
    std::cout << "\n2. Testing get_compression_name():" << std::endl;
    std::cout << "  HEVC: " << get_compression_name(ExtendedCompressionType::HEVC) << std::endl;
    std::cout << "  AV1: " << get_compression_name(ExtendedCompressionType::AV1) << std::endl;
    std::cout << "  DEFLATE: " << get_compression_name(ExtendedCompressionType::DEFLATE) << std::endl;
    
    // Test is_lossless_compression
    std::cout << "\n3. Testing is_lossless_compression():" << std::endl;
    std::cout << "  HEVC is lossless: " << (is_lossless_compression(ExtendedCompressionType::HEVC) ? "Yes" : "No") << std::endl;
    std::cout << "  DEFLATE is lossless: " << (is_lossless_compression(ExtendedCompressionType::DEFLATE) ? "Yes" : "No") << std::endl;
    
    // Test to_heif_compression_format
    std::cout << "\n4. Testing to_heif_compression_format():" << std::endl;
    auto hevc_fmt = to_heif_compression_format(ExtendedCompressionType::HEVC);
    auto av1_fmt = to_heif_compression_format(ExtendedCompressionType::AV1);
    std::cout << "  HEVC format code: " << static_cast<int>(hevc_fmt) << std::endl;
    std::cout << "  AV1 format code: " << static_cast<int>(av1_fmt) << std::endl;
    
    // Test to_unci_compression
    std::cout << "\n5. Testing to_unci_compression():" << std::endl;
    auto deflate_unci = to_unci_compression(ExtendedCompressionType::DEFLATE);
    auto zlib_unci = to_unci_compression(ExtendedCompressionType::ZLIB);
    std::cout << "  DEFLATE unci code: " << static_cast<int>(deflate_unci) << std::endl;
    std::cout << "  ZLIB unci code: " << static_cast<int>(zlib_unci) << std::endl;
    
    std::cout << "\n✓ All helper function tests passed" << std::endl;
}

// test_helper_functions();

#endif // HEIF_TEST_HELPERS_H

In [ ]:
// test_helper_functions();


# Working POINT 2

## List Available Features

### Detect TILI Support at Runtime

In [ ]:
#ifndef HEIF_TILI_RUNTIME_CHECK_H
#define HEIF_TILI_RUNTIME_CHECK_H

bool is_tili_actually_supported() {
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
    // Create a minimal test context to check if TILI is available
    heif_context* test_ctx = heif_context_alloc();
    
    heif_tiled_image_parameters test_params;
    memset(&test_params, 0, sizeof(test_params));
    test_params.version = 1;
    test_params.image_width = 512;
    test_params.image_height = 512;
    test_params.tile_width = 256;
    test_params.tile_height = 256;
    //test_params.offset_field_length = 32;
    //test_params.size_field_length = 24;
    //test_params.tiles_are_sequential = 1;
    
    // Get a dummy encoder
    heif_encoder* test_encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(
        test_ctx, heif_compression_HEVC, &test_encoder);
    
    if (error.code != heif_error_Ok) {
        heif_context_free(test_ctx);
        std::cout << "TILI check: Cannot get encoder" << std::endl;
        return false;
    }
    
    // Try to create TILI image
    heif_image_handle* test_handle = nullptr;
    error = heif_context_add_tiled_image(test_ctx, &test_params, nullptr,
                                        test_encoder, &test_handle);
    
    bool supported = (error.code == heif_error_Ok || 
                     error.code == heif_error_Usage_error); // Usage error is OK, means API exists
    
    // Check for specific "unsupported" errors
    if (error.code == heif_error_Unsupported_filetype ||
        error.code == heif_error_Unsupported_feature) {
        supported = false;
    }
    
    if (test_handle) {
        heif_image_handle_release(test_handle);
    }
    heif_encoder_release(test_encoder);
    heif_context_free(test_ctx);
    
    if (!supported) {
        std::cout << "TILI check: " << error.message << std::endl;
        std::cout << "TILI is NOT supported in this libheif build" << std::endl;
    } else {
        std::cout << "TILI check: TILI appears to be supported" << std::endl;
    }
    
    return supported;
#else
    std::cout << "TILI check: Experimental features not enabled" << std::endl;
    return false;
#endif
}

// Cache the result
inline bool check_tili_support_cached() {
    static bool checked = false;
    static bool supported = false;
    
    if (!checked) {
        std::cout << "\n=== Checking TILI Support ===" << std::endl;
        supported = is_tili_actually_supported();
        checked = true;
        std::cout << "TILI Support: " << (supported ? "YES ✓" : "NO ✗") << std::endl;
        std::cout << std::string(40, '=') << "\n" << std::endl;
    }
    
    return supported;
}

#endif // HEIF_TILI_RUNTIME_CHECK_H

In [ ]:
// if (!check_tili_support_cached()) {
//     std::cout << "TILI not supported in this libheif build." << std::endl;
//     std::cout << "Falling back to Grid scheme..." << std::endl;
// }
// else {
//     std::cout << "TILI is supported. Using TILI tiling scheme." << std::endl;
// } 

### Compressions

In [ ]:
// // Display all available compression algorithms
// list_all_compression_algorithms();

// // Display available encoders
// std::cout << "\n=== Available Encoders ===\n" << std::endl;
// list_encoders(heif_compression_HEVC);
// list_encoders(heif_compression_AV1);
// list_encoders(heif_compression_uncompressed);

### Tiling Schemes

In [ ]:
// List capabilities
list_tiling_schemes();


## Quick Start Example - Main Encoding Examples

In [ ]:
// // Example 1: Default - memory-efficient (existing behavior)
// {
//     PyramidConfig config;
//     config.num_levels = 5;
//     config.use_tiling = true;
//     config.tiling_scheme = TilingScheme::TILI;
//     config.tile_width = 256;
//     config.tile_height = 256;
//     config.compression = ExtendedCompressionType::DEFLATE;
//     config.write_order = PyramidWriteOrder::Default;  // Or omit (default)
    
//     encode_pyramid_with_config(input_file, "output_default.heif", config);
// }

// // Example 2: COG-style (coarse first)
// {
//     PyramidConfig config;
//     config.num_levels = 5;
//     config.use_tiling = true;
//     config.tiling_scheme = TilingScheme::TILI;
//     config.tile_width = 256;
//     config.tile_height = 256;
//     config.compression = ExtendedCompressionType::DEFLATE;
//     config.write_order = PyramidWriteOrder::CoarseFirst;  // COG-style
    
//     encode_pyramid_with_config(input_file, "output_cog.heif", config);
// }

// // Example 3: With padding metadata in both formats
// {
//     PyramidConfig config;
//     config.num_levels = 5;
//     config.use_tiling = true;
//     config.tiling_scheme = TilingScheme::TILI;
//     config.tile_width = 256;
//     config.tile_height = 256;
//     config.compression = ExtendedCompressionType::DEFLATE;
//     config.write_order = PyramidWriteOrder::CoarseFirst;
//     config.store_padding_metadata = true;
//     config.padding_format = PaddingMetadataFormat::BOTH;
    
//     encode_pyramid_with_config(input_file, "output_cog_metadata.heif", config);
// }


Uncomment and modify the following to encode your GeoTIFF:

In [ ]:
{
std::cout << "Quick start example:" << std::endl;
// define some variables for input and output paths
std::string inputFileStr = (fs::current_path() / "srcdata" / "ACT2017.tiff").string();
std::string outputDirStr = (fs::current_path() / "benchmark_output").string();
std::string outputFileStr = (fs::current_path() / "benchmark_output" / "output.heif").string();
const char* input_file = inputFileStr.c_str();
const char* output_file = outputFileStr.c_str();
const char* output_dir = outputDirStr.c_str();

if (!fs::exists(std::string(input_file))) {
    std::cout << "ERROR: " << std::string(input_file) << " Path not found.\n";
} else {
    std::cout << std::string(input_file) << " File or directory exists!\n";
}
// Uncomment and set your input file path:
// const char* input_file = "path/to/your/input.tif";
//std::cout << "Current working directory: " << fs::current_path() << std::endl;
//std::cout << "Looking for input file in: " << (fs::current_path() / "srcdata" / "ACT2017.tiff") << std::endl;
// 1. Store the string in a local variable first
//fs::path myPath = fs::current_path() / "srcdata" / "ACT2017.tiff";
//std::string pathStr = myPath.string(); 

// 2. Get the pointer from that variable
//const char* cStr = pathStr.c_str();

//std::cout << "Path as C-string: " << cStr << std::endl;


// Example 1: Single image with HEVC
// encode_example(input_file, "output.heif", ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_hevc_70.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_example(input_file, output_file, ExtendedCompressionType::HEVC, 70);

// Example 2: Single image with AV1
// encode_example(input_file, "output.avif", ExtendedCompressionType::AV1, 60);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_av1_60.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_example(input_file, output_file, ExtendedCompressionType::AV1, 60);

// Example 3: Lossless with DEFLATE
// encode_nontiled(input_file, "output_lossless.heif", ExtendedCompressionType::DEFLATE);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_lossless_deflate.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);

// Example 4: Tiled image
// encode_tiled_grid(input_file, "output_tiled.heif", 512, 512, 
//                   ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_512_hevc_70.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_tiled_grid(input_file, output_file, 512, 512, 
//                  ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_256_deflate_100.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_tiled_grid(input_file, output_file, 256, 256, 
//                  ExtendedCompressionType::DEFLATE, 100);

// Example 5: Run basic demo (creates multiple output files)
// demo_basic_encoding(input_file, output_dir);
// demo_basic_encoding(input_file, output_dir);

// Example 6: Run all compression demos (creates many output files)
// demo_all_compressions(input_file, output_dir);
// demo_all_compressions(input_file, output_dir);

// Example 7: Test all tiling schemes
// test_all_tiling_schemes(input_file, output_dir);
// test_all_tiling_schemes(input_file, output_dir);


// Example 8: Create image with unci tiling scheme 
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_unci_256_deflate_100.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// std::cout << "Output file pointer: " << output_file << std::endl;
// encode_tiled_unci(input_file, output_file, 256, 256, 
//                  ExtendedCompressionType::DEFLATE, 100);

// Example 9: Create image with tili tiling scheme (experimental, may not be supported by all decoders)
//outputFileStr =  (fs::current_path() / "benchmark_output" / "output_tili_512_deflate.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// encode_tiled_tili(input_file, output_file, 512, 512,
//              ExtendedCompressionType::DEFLATE);

// Exmapele 10: Create image with tili tiling scheme using ecnode_tiled and specifying TILI scheme
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate2.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE);

// Example 11: Create tiled image with tili tiling scheme and HEVC compression using encode_tiled and specifying TILI scheme
// outputFileStr = (fs::current_path() / "benchmark_output" /  "output_tili_512_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::HEVC, 70);

// Example 12: Create a tiled with TILI and AV1 compression (may not be supported in all implementations, encoder may report errors)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_av1_60.avif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::AV1, 60);

// Example 13: Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// Test with DEFLATE (should work)
// std::cout << "\n   Testing TILI with DEFLATE (lossless)..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE);

// // offset and size with tpad - Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// // Test with DEFLATE (should work)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(32, 24, false);
// // OffsetTableConfig config1(32, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tconfig1);

// // offset and size without tpad - Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate_nopad.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(32, 24, false);
// // OffsetTableConfig config1(32, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = false;  // DISABLE
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tconfig1);

// // offset only with tpad - Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate_offsetonly.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(40, 0, true);
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tconfig1);

// // size only with tpad - Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate_sizeonly.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(0, 32, true);
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tconfig1);
    
// Example 14: Create a tiled image with TILI tiling scheme and uncompressed (lossless, should work in all implementations that support TILI)
// Test with uncompressed (should work)
//std::cout << "\n   Testing TILI with uncompressed..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_uncompressed.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::Uncompressed);

// Example 15: Create pyramids with HEVC compression
// Test pyramids
// std::cout << "\n Pyramid Tests:" << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_pyramid_3levels_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_pyramid(input_file, output_file, 3,
//                 ExtendedCompressionType::HEVC, 70);

// Example 16: Create pyramids with AV1 compression (may not be supported in all implementations, encoder may report errors)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_pyramid_5levels_av1_60.avif").string();
// output_file = outputFileStr.c_str();
// encode_pyramid(input_file, output_file, 5,
//                 ExtendedCompressionType::AV1, 60);

// Example 17: Create tiled pyramids with different tiling schemes and compressions (some combinations may not be supported, encoder may report errors)
// Test tiled pyramids
// std::cout << "\nTiled Pyramid Tests:" << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_grid_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::Grid,
//                     ExtendedCompressionType::HEVC, 70);

// Example 18: Create tiled pyramid with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_tili_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::TILI,
//                     ExtendedCompressionType::DEFLATE);

// Example 18: Create tiled pyramid with TILI tiling scheme and UNCOMPRESSED) - works after memmory cleaning after each level processing and padding tile to exact same tile size for TILI
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_tili_uncompressed.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::TILI,
//                     ExtendedCompressionType::Uncompressed);
// // Example 19: Create tiled pyramid with TILI tiling scheme and DEFLATE compression but with smaller tile size (256x256) - works after memmory cleaning after each level processing and padding tile to exact same tile size for TILI
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255);

// // No tpad written out
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate_nopad.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(32, 24, false);
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.store_padding_metadata = false;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);

// // writing order with COG-style coarse first
// // Write out with configuration on both padding formats
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate_no_pad_order.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(32, 24, false);
// // OffsetTableConfig config1(32, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// PyramidWriteOrder wconfig1(PyramidWriteOrder::CoarseFirst); // COG-style coarse first
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1,wconfig1);

// // Write out with configuration on both padding formats
outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate_order.heif").string();
output_file = outputFileStr.c_str();
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
OffsetTableConfig config1(48, 24, false);
// OffsetTableConfig config1(32, 0, true);
TilingConfig tconfig1;
tconfig1.padding_format = PaddingMetadataFormat::BOTH;
tconfig1.store_padding_metadata = true;  // enable
PyramidWriteOrder wconfig1(PyramidWriteOrder::Default); //::CoarseFirst); // COG-style coarse first
encode_tiled_pyramid(input_file, output_file,
                     5, 256, 256, TilingScheme::TILI,
                     ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tconfig1, wconfig1);


// // Write out with configuration on both padding formats
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// // OffsetTableConfig config1(32, 24, false);
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);

// // Write out - offset (only offset, sequential) with configuration on both padding formats
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate_offsetonly.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(32, 0, true);
// // OffsetTableConfig config1(32, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);

// // Write out - size (only szie, sequential) with configuration on both padding formats
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate_sizeonly.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(0, 24, true);
// // OffsetTableConfig config1(32, 0, true);
// TilingConfig tconfig1;
// tconfig1.padding_format = PaddingMetadataFormat::BOTH;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);

// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetOnly);
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1);

// // Example 20: Create tiled pyramid with UNCI tiling scheme and DEFLATE compression but with smaller tile size (256x256)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_unci_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(40, 32, true); // no effect for unci since libheif does expose offset and size fields for unci, but they are not actually used to write out offset tables since unci spec does not support them
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.store_padding_metadata = true;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::UNCI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);


// // No tpad written out
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_unci_256_deflate_nopad.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(40, 32, false); // no effect for unci since libheif does expose offset and size fields for unci, but they are not actually used to write out offset tables since unci spec does not support them
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.store_padding_metadata = false;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::UNCI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);


// // Example 21: Create tiled pyramid with GRID tiling scheme and DEFLATE compression but with smaller tile size (256x256)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_grid_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(40, 32, false); // no effect for grid since no offset table is written out
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.store_padding_metadata = true;  
// encode_tiled_pyramid(input_file, output_file,
//                     5, 256, 256, TilingScheme::Grid,
//                     ExtendedCompressionType::DEFLATE,
//                     100, 8, 255,config1,tconfig1);

// // No tpad written out
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_grid_256_deflate_nopad.heif").string();
// output_file = outputFileStr.c_str();
// // OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// OffsetTableConfig config1(40, 32, false); // no effect for grid since no offset table is written out
// // OffsetTableConfig config1(40, 0, true);
// TilingConfig tconfig1;
// tconfig1.store_padding_metadata = false;  // DISABLE
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::Grid,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1,tconfig1);



// Example 13: Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// Test with DEFLATE (should work)
// std::cout << "\n   Testing Grid tiling with DEFLATE (lossless)..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_grid_512_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::Grid, ExtendedCompressionType::DEFLATE);

// Example 14: Creating a tiled image with TILI tiling scheme and DEFLATE compression plus custom padding info
// {
//     outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
//     output_file = outputFileStr.c_str();
//     OffsetTableConfig config1(40, 32, false);
//     TilingConfig tiling_cfg;  // Use defaults (store metadata)
//     encode_tiled_pyramid(input_file, output_file,
//                         5, 256, 256, TilingScheme::TILI,
//                         ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, tiling_cfg);
// }

// std::cout << "\nReady to encode! Uncomment examples above and set your input file path." << std::endl;
// }

// Example 15: Creating a tiled image with TILI tiling scheme and DEFLATE compression but without custom padding info
// {
//     TilingConfig no_metadata;
//     no_metadata.store_padding_metadata = false;
//     encode_tiled_pyramid(input_file, output_file,
//                         5, 256, 256, TilingScheme::TILI,
//                         ExtendedCompressionType::DEFLATE, 100, 8, 255, config1, no_metadata);
// }

}

## Usage Examples Encoding with Compact Padding Metadata

In [ ]:
#ifndef HEIF_EXAMPLE_COMPACT_METADATA_H
#define HEIF_EXAMPLE_COMPACT_METADATA_H

void example_encode_with_compact_metadata() {
    std::string input = "input.tif";
    std::string output = "output_with_metadata.heif";
    
    std::cout << "\n=== Encoding with Compact Padding Metadata ===" << std::endl;
    
    // Encode tiled image with padding
    encode_tiled_tili(input.c_str(), output.c_str(), 
                     256, 256,
                     ExtendedCompressionType::DEFLATE, 
                     50, 8, 0);  // padding_value = 0
    
    std::cout << "\n=== Verifying Metadata ===" << std::endl;
    
    // Read back and verify
    auto metadata_list = TpadMetadataReader::read_all_levels(output.c_str());
    
    if (metadata_list) {
        std::cout << "✓ Found " << metadata_list->size() << " level(s) of padding metadata" << std::endl;
        
        for (size_t i = 0; i < metadata_list->size(); i++) {
            std::cout << "\n--- Level " << i << " ---" << std::endl;
            const auto& meta = (*metadata_list)[i];
            meta.print();
            meta.print_tile_map();
            
            auto stats = meta.compute_statistics();
            stats.print();
        }
        
        // Process tiles
        process_tiles_with_tpad(output.c_str());
    } else {
        std::cerr << "⚠ No padding metadata found" << std::endl;
    }
}

void example_encode_pyramid_with_metadata() {
    std::string input = "large_image.tif";
    std::string output = "pyramid_with_metadata.heif";
    
    std::cout << "\n=== Encoding Pyramid with Per-Level Metadata ===" << std::endl;
    
    // Configure pyramid with tiling
    PyramidConfig config;
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::TILI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::DEFLATE;
    config.quality = 100;
    config.output_bit_depth = 8;
    config.padding_value = 0;
    
    encode_pyramid_with_config(input.c_str(), output.c_str(), config);
    
    std::cout << "\n=== Verifying Pyramid Metadata ===" << std::endl;
    
    auto metadata_list = TpadMetadataReader::read_all_levels(output.c_str());
    
    if (metadata_list) {
        std::cout << "✓ Found padding metadata for " << metadata_list->size() 
                  << " pyramid level(s)" << std::endl;
        
        // Analyze each level
        for (size_t level = 0; level < metadata_list->size(); level++) {
            std::cout << "\n========================================" << std::endl;
            std::cout << "PYRAMID LEVEL " << level << std::endl;
            std::cout << "========================================" << std::endl;
            
            const auto& meta = (*metadata_list)[level];
            meta.print();
            
            std::cout << "\nComputed dimensions:" << std::endl;
            std::cout << "  Original: " << meta.original_width 
                      << "x" << meta.original_height << std::endl;
            std::cout << "  Padded: " << meta.get_padded_width() 
                      << "x" << meta.get_padded_height() << std::endl;
            std::cout << "  Tiles: " << meta.get_num_cols() 
                      << "x" << meta.get_num_rows() << std::endl;
            
            auto stats = meta.compute_statistics();
            stats.print();
        }
    }
}

// Compare storage overhead
void compare_metadata_formats() {
    std::cout << "\n=== Metadata Format Comparison ===" << std::endl;
    std::cout << "\nFor 4096x4096 image with 256x256 tiles (256 tiles):\n" << std::endl;
    
    struct FormatInfo {
        std::string name;
        size_t size;
        std::string description;
    };
    
    std::vector<FormatInfo> formats = {
        {"Compact binary", 24, "Fixed size, compute on-the-fly"},
        {"Per-tile binary", 2080, "32 byte header + 8 bytes/tile"},
        {"XMP metadata", 500, "XML with global dimensions only"},
        {"Full XMP", 15000, "XML with per-tile info (if stored)"}
    };
    
    std::cout << std::left 
              << std::setw(20) << "Format" 
              << std::setw(12) << "Size"
              << std::setw(50) << "Description" << std::endl;
    std::cout << std::string(82, '-') << std::endl;
    
    for (const auto& fmt : formats) {
        std::cout << std::left
                  << std::setw(20) << fmt.name
                  << std::setw(12) << (std::to_string(fmt.size) + " bytes")
                  << std::setw(50) << fmt.description << std::endl;
    }
    
    std::cout << "\n✓ Compact binary is the clear winner!" << std::endl;
    std::cout << "  - Only 24 bytes regardless of tile count" << std::endl;
    std::cout << "  - Pure C++ parsing (no XML library)" << std::endl;
    std::cout << "  - Fast computation of per-tile info" << std::endl;
}

std::cout << "Example functions with compact metadata defined." << std::endl;

#endif // HEIF_EXAMPLE_COMPACT_METADATA_H

In [ ]:
// // Example 1: Read and display metadata
// TpadMetadataReader::display_all_metadata("output.heif");

// // Example 2: Process tiles with metadata
// process_tiles_with_tpad("output.heif");

// // Example 3: Just read metadata programmatically
// auto metadata = TpadMetadataReader::read_all_levels("output.heif");
// if (metadata) {
//     for (size_t i = 0; i < metadata->size(); i++) {
//         const auto& meta = (*metadata)[i];
//         std::cout << "Level " << i << ": " 
//                   << meta.original_width << "x" << meta.original_height 
//                   << std::endl;
//     }
// }

In [ ]:
// // Example 1: 
// {
//     std::string input = "srcdata/ACT2017.tiff";
//     std::string output = "output/output_with_metadata.heif";
    
//     std::cout << "\n=== Encoding with Compact Padding Metadata ===" << std::endl;
    
//     // Encode tiled image with padding
//     encode_tiled_tili(input.c_str(), output.c_str(), 
//                      1024, 1024,
//                      ExtendedCompressionType::DEFLATE, 
//                      50, 8, 0);  // padding_value = 0
    
//     std::cout << "\n=== Verifying Metadata ===" << std::endl;
    
//     // Read back and verify
//     auto metadata_list = TpadMetadataReader::read_all_levels(output.c_str());
    
//     if (metadata_list) {
//         std::cout << "✓ Found " << metadata_list->size() << " level(s) of padding metadata" << std::endl;
        
//         for (size_t i = 0; i < metadata_list->size(); i++) {
//             std::cout << "\n--- Level " << i << " ---" << std::endl;
//             const auto& meta = (*metadata_list)[i];
//             meta.print();
//             meta.print_tile_map();
            
//             auto stats = meta.compute_statistics();
//             stats.print();
//         }
        
//         // Process tiles
//         // process_tiles_with_tpad(output.c_str()); // decoder not fixed - will use decoder for tili/grid/unci in readers
//     } else {
//         std::cerr << "⚠ No padding metadata found" << std::endl;
//     }
// }


// // encode with tiled, pyramidal
// {
//     std::string input = "srcdata/ACT2017.tiff";
//     std::string output = "output/pyramid_with_metadata.heif";
    
//     std::cout << "\n=== Encoding Pyramid with Per-Level Metadata ===" << std::endl;
    
//     // Configure pyramid with tiling
//     PyramidConfig config;
//     config.num_levels = 5;
//     config.use_tiling = true;
//     config.tiling_scheme = TilingScheme::TILI;
//     config.tile_width = 256;
//     config.tile_height = 256;
//     config.compression = ExtendedCompressionType::DEFLATE;
//     config.quality = 100;
//     config.output_bit_depth = 8;
//     config.padding_value = 0;
    
//     encode_pyramid_with_config(input.c_str(), output.c_str(), config);
    
//     std::cout << "\n=== Verifying Pyramid Metadata ===" << std::endl;
    
//     auto metadata_list = TpadMetadataReader::read_all_levels(output.c_str());
    
//     if (metadata_list) {
//         std::cout << "✓ Found padding metadata for " << metadata_list->size() 
//                   << " pyramid level(s)" << std::endl;
        
//         // Analyze each level
//         for (size_t level = 0; level < metadata_list->size(); level++) {
//             std::cout << "\n========================================" << std::endl;
//             std::cout << "PYRAMID LEVEL " << level << std::endl;
//             std::cout << "========================================" << std::endl;
            
//             const auto& meta = (*metadata_list)[level];
//             meta.print();
            
//             std::cout << "\nComputed dimensions:" << std::endl;
//             std::cout << "  Original: " << meta.original_width 
//                       << "x" << meta.original_height << std::endl;
//             std::cout << "  Padded: " << meta.get_padded_width() 
//                       << "x" << meta.get_padded_height() << std::endl;
//             std::cout << "  Tiles: " << meta.get_num_cols() 
//                       << "x" << meta.get_num_rows() << std::endl;
            
//             auto stats = meta.compute_statistics();
//             stats.print();
//         }
//     }
// }


## Usage Examples with Tile Padding Info configured

In [ ]:
#ifndef HEIF_USAGE_TPAD_CONFIG_H
#define HEIF_USAGE_TPAD_CONFIG_H

void demonstrate_tpad_config_options() {
    const char* input = "input.tif";
    
    // Example 1: Default behavior (store binary metadata)
    {
        std::cout << "\n=== Example 1: Default (store tpad binary) ===" << std::endl;
        encode_tiled_tili(input, "output_default.heif", 256, 256,
                         ExtendedCompressionType::DEFLATE);
        // Will store 24-byte tpad metadata
    }
    
    // Example 2: Disable padding metadata
    {
        std::cout << "\n=== Example 2: No padding metadata ===" << std::endl;
        TilingConfig config;
        config.store_padding_metadata = false;  // DISABLE
        
        encode_tiled(input, "output_no_metadata.heif", 256, 256,
                    TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
                    50, 8, 0, OffsetTableConfig(), config);
        // No tpad metadata written
    }
    
    // Example 3: Store as XMP instead of binary
    {
        std::cout << "\n=== Example 3: XMP format ===" << std::endl;
        TilingConfig config;
        config.padding_format = PaddingMetadataFormat::XMP_TEXT;
        
        encode_tiled(input, "output_xmp_metadata.heif", 256, 256,
                    TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
                    50, 8, 0, OffsetTableConfig(), config);
        // Stores XMP instead of binary tpad
    }
    
    // Example 4: Store both formats
    {
        std::cout << "\n=== Example 4: Both tpad and XMP ===" << std::endl;
        TilingConfig config;
        config.padding_format = PaddingMetadataFormat::BOTH;
        
        encode_tiled(input, "output_both_metadata.heif", 256, 256,
                    TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
                    50, 8, 0, OffsetTableConfig(), config);
        // Stores both tpad and XMP
    }
    
    // Example 5: Pyramid with configurable metadata
    {
        std::cout << "\n=== Example 5: Pyramid without metadata ===" << std::endl;
        PyramidConfig config;
        config.num_levels = 5;
        config.use_tiling = true;
        config.tiling_scheme = TilingScheme::TILI;
        config.tile_width = 256;
        config.tile_height = 256;
        config.compression = ExtendedCompressionType::DEFLATE;
        config.store_padding_metadata = false;  // DISABLE for pyramid
        
        encode_pyramid_with_config(input, "output_pyramid_no_metadata.heif", config);
        // No tpad metadata for any level
    }
    
    // Example 6: Pyramid with XMP metadata
    {
        std::cout << "\n=== Example 6: Pyramid with XMP metadata ===" << std::endl;
        PyramidConfig config;
        config.num_levels = 5;
        config.use_tiling = true;
        config.tiling_scheme = TilingScheme::TILI;
        config.tile_width = 256;
        config.tile_height = 256;
        config.compression = ExtendedCompressionType::DEFLATE;
        config.store_padding_metadata = true;
        config.padding_format = PaddingMetadataFormat::XMP_TEXT;
        
        encode_pyramid_with_config(input, "output_pyramid_xmp.heif", config);
        // XMP metadata for each level
    }
}

#endif // HEIF_USAGE_TPAD_CONFIG_H

## Helper function to print out all TPAD configuration options

In [ ]:
#ifndef HEIF_PRINT_TPAD_CONFIG_H
#define HEIF_PRINT_TPAD_CONFIG_H

void print_tpad_configuration_options() {
    std::cout << "\n╔════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Tile Padding Metadata Configuration Options              ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════╝\n" << std::endl;
    
    std::cout << "Configuration Options:" << std::endl;
    std::cout << "  • store_padding_metadata (bool)" << std::endl;
    std::cout << "    - true:  Store padding metadata (default)" << std::endl;
    std::cout << "    - false: Skip padding metadata" << std::endl;
    
    std::cout << "\n  • padding_format (enum)" << std::endl;
    std::cout << "    - TPAD_BINARY: 24-byte binary format (default)" << std::endl;
    std::cout << "      ✓ Most efficient (24 bytes)" << std::endl;
    std::cout << "      ✓ Fast parsing" << std::endl;
    std::cout << "      ✓ No XML overhead" << std::endl;
    
    std::cout << "\n    - XMP_TEXT: XML metadata format" << std::endl;
    std::cout << "      ✓ Human-readable" << std::endl;
    std::cout << "      ✓ Compatible with XMP tools" << std::endl;
    std::cout << "      ✗ Larger file size (~500 bytes)" << std::endl;
    
    std::cout << "\n    - BOTH: Store both formats" << std::endl;
    std::cout << "      ✓ Maximum compatibility" << std::endl;
    std::cout << "      ✗ Redundant storage" << std::endl;
    
    std::cout << "\nRecommendation:" << std::endl;
    std::cout << "  Use TPAD_BINARY for production (most efficient)" << std::endl;
    std::cout << "  Use XMP_TEXT for debugging/inspection" << std::endl;
    std::cout << "  Use BOTH only if maximum compatibility needed" << std::endl;
    
    std::cout << "\n" << std::string(62, '=') << std::endl;
}

#endif // HEIF_PRINT_TPAD_CONFIG_H

### Tests

In [ ]:
#ifndef HEIF_MULTILEVEL_TPAD_READER_H
#define HEIF_MULTILEVEL_TPAD_READER_H

class MultiLevelTpadReader {
public:
    // Try multiple strategies to find the metadata
    static std::optional<MultiLevelPaddingMetadata> read(const char* filename) {
        heif_context* ctx = heif_context_alloc();
        heif_error error = heif_context_read_from_file(ctx, filename, nullptr);
        
        if (error.code != heif_error_Ok) {
            std::cerr << "Failed to open file: " << error.message << std::endl;
            heif_context_free(ctx);
            return std::nullopt;
        }
        
        std::cout << "\n=== Searching for Multi-Level Padding Metadata ===\n" << std::endl;
        
        // Strategy 1: Check primary image
        std::cout << "Strategy 1: Checking primary image..." << std::endl;
        auto result = read_from_primary_image(ctx);
        if (result) {
            std::cout << "✓ Found metadata on primary image\n" << std::endl;
            heif_context_free(ctx);
            return result;
        }
        
        // Strategy 2: Check all top-level images
        std::cout << "Strategy 2: Checking all top-level images..." << std::endl;
        result = read_from_all_images(ctx);
        if (result) {
            std::cout << "✓ Found metadata on image item\n" << std::endl;
            heif_context_free(ctx);
            return result;
        }
        
        // Strategy 3: Check file-level metadata (if API supports)
        std::cout << "Strategy 3: Checking file-level metadata..." << std::endl;
        // Note: libheif may not expose file-level metadata directly
        // This would require direct box parsing
        
        std::cout << "✗ No metadata found\n" << std::endl;
        heif_context_free(ctx);
        return std::nullopt;
    }
    
private:
    static std::optional<MultiLevelPaddingMetadata> read_from_primary_image(heif_context* ctx) {
        heif_image_handle* handle;
        heif_error error = heif_context_get_primary_image_handle(ctx, &handle);
        
        if (error.code != heif_error_Ok) {
            std::cout << "  Cannot get primary image handle: " << error.message << std::endl;
            return std::nullopt;
        }
        
        auto result = read_from_handle(handle);
        heif_image_handle_release(handle);
        return result;
    }
    
    static std::optional<MultiLevelPaddingMetadata> read_from_all_images(heif_context* ctx) {
        int num_images = heif_context_get_number_of_top_level_images(ctx);
        std::vector<heif_item_id> ids(num_images);
        heif_context_get_list_of_top_level_image_IDs(ctx, ids.data(), num_images);
        
        std::cout << "  Checking " << num_images << " top-level images..." << std::endl;
        
        for (int i = 0; i < num_images; i++) {
            heif_image_handle* handle;
            heif_error error = heif_context_get_image_handle(ctx, ids[i], &handle);
            
            if (error.code == heif_error_Ok) {
                std::cout << "    Image " << i << " (ID: " << ids[i] << ")..." << std::endl;
                auto result = read_from_handle(handle);
                heif_image_handle_release(handle);
                
                if (result) {
                    std::cout << "    ✓ Found metadata on image " << i << std::endl;
                    return result;
                }
            }
        }
        
        return std::nullopt;
    }
    
    static std::optional<MultiLevelPaddingMetadata> read_from_handle(heif_image_handle* handle) {
        // Look for "mtpd" metadata
        heif_item_id mtpd_ids[10];
        int num_mtpd = heif_image_handle_get_list_of_metadata_block_IDs(
            handle, "mtpd", mtpd_ids, 10);
        
        if (num_mtpd > 0) {
            std::cout << "      Found " << num_mtpd << " 'mtpd' block(s)" << std::endl;
        }
        
        for (int i = 0; i < num_mtpd; i++) {
            size_t metadata_size = heif_image_handle_get_metadata_size(handle, mtpd_ids[i]);
            std::cout << "      Block " << i << ": " << metadata_size << " bytes" << std::endl;
            
            // Validate size (must be 8 + 24*n)
            if (metadata_size < 8) {
                std::cout << "        ✗ Too small for header" << std::endl;
                continue;
            }
            
            if ((metadata_size - 8) % 24 != 0) {
                std::cout << "        ✗ Invalid size (not 8 + 24*n)" << std::endl;
                continue;
            }
            
            std::vector<uint8_t> metadata_buffer(metadata_size);
            heif_error error = heif_image_handle_get_metadata(
                handle, mtpd_ids[i], metadata_buffer.data());
            
            if (error.code == heif_error_Ok) {
                // Verify magic
                uint32_t magic;
                memcpy(&magic, metadata_buffer.data(), 4);
                std::cout << "        Magic: 0x" << std::hex << magic << std::dec << std::endl;
                
                auto result = MultiLevelPaddingMetadata::deserialize(
                    metadata_buffer.data(), metadata_size);
                
                if (result) {
                    std::cout << "        ✓ Successfully deserialized" << std::endl;
                    return result;
                } else {
                    std::cout << "        ✗ Deserialization failed" << std::endl;
                }
            } else {
                std::cout << "        ✗ Failed to read metadata: " << error.message << std::endl;
            }
        }
        
        return std::nullopt;
    }
};

// Convenience function for display
void display_multilevel_metadata(const char* filename) {
    std::cout << "\n" << std::string(60, '=') << std::endl;
    std::cout << "Reading Multi-Level Padding Metadata" << std::endl;
    std::cout << "File: " << filename << std::endl;
    std::cout << std::string(60, '=') << std::endl;
    
    auto metadata = MultiLevelTpadReader::read(filename);
    
    if (metadata) {
        std::cout << "\n✓ Successfully read metadata!\n" << std::endl;
        metadata->print();
        
        std::cout << "\n=== Summary ===" << std::endl;
        std::cout << "Total pyramid levels with padding: " << metadata->num_levels << std::endl;
        std::cout << "Total metadata size: " << (8 + 24 * metadata->num_levels) << " bytes" << std::endl;
        
        for (size_t i = 0; i < metadata->levels.size(); i++) {
            const auto& level = metadata->levels[i];
            std::cout << "\nLevel " << i << ":" << std::endl;
            std::cout << "  Original: " << level.original_width << "x" << level.original_height << std::endl;
            std::cout << "  Padded: " << level.get_padded_width() << "x" << level.get_padded_height() << std::endl;
            std::cout << "  Tiles: " << level.get_num_cols() << "x" << level.get_num_rows() 
                      << " (" << level.get_total_tiles() << " total)" << std::endl;
        }
    } else {
        std::cout << "\n✗ No metadata found or failed to read\n" << std::endl;
    }
    
    std::cout << "\n" << std::string(60, '=') << "\n" << std::endl;
}

#endif // HEIF_MULTILEVEL_TPAD_READER_H

In [ ]:
// print_tpad_configuration_options()

In [ ]:
// Test reading the metadata
// display_multilevel_metadata("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif");

### Debug

In [ ]:
#ifndef DEBUG_READ_ALL_METADATA_H
#define DEBUG_READ_ALL_METADATA_H
void debug_all_metadata(const char* filename) {
    heif_context* ctx = heif_context_alloc();
    heif_error error = heif_context_read_from_file(ctx, filename, nullptr);
    
    if (error.code != heif_error_Ok) {
        std::cerr << "Failed to open file" << std::endl;
        heif_context_free(ctx);
        return;
    }
    
    std::cout << "\n=== ALL METADATA IN FILE ===\n" << std::endl;
    
    int num_images = heif_context_get_number_of_top_level_images(ctx);
    std::vector<heif_item_id> ids(num_images);
    heif_context_get_list_of_top_level_image_IDs(ctx, ids.data(), num_images);
    
    std::cout << "Total top-level images: " << num_images << "\n" << std::endl;
    
    for (int i = 0; i < num_images; i++) {
        heif_image_handle* handle;
        error = heif_context_get_image_handle(ctx, ids[i], &handle);
        
        if (error.code != heif_error_Ok) continue;
        
        std::cout << "Image " << i << " (ID: " << ids[i] << "):" << std::endl;
        
        // Get all metadata types
        int num_metadata = heif_image_handle_get_number_of_metadata_blocks(handle, nullptr);
        std::cout << "  Total metadata blocks: " << num_metadata << std::endl;
        
        if (num_metadata > 0) {
            std::vector<heif_item_id> metadata_ids(num_metadata);
            heif_image_handle_get_list_of_metadata_block_IDs(handle, nullptr, 
                                                             metadata_ids.data(), num_metadata);
            
            for (int j = 0; j < num_metadata; j++) {
                const char* type = heif_image_handle_get_metadata_type(handle, metadata_ids[j]);
                const char* content_type = heif_image_handle_get_metadata_content_type(handle, metadata_ids[j]);
                size_t size = heif_image_handle_get_metadata_size(handle, metadata_ids[j]);
                
                std::cout << "    Block " << j << ":" << std::endl;
                std::cout << "      Type: " << (type ? type : "(null)") << std::endl;
                std::cout << "      Content-Type: " << (content_type ? content_type : "(null)") << std::endl;
                std::cout << "      Size: " << size << " bytes" << std::endl;
                
                // If it's mtpd, try to read it
                if (type && std::string(type) == "mtpd") {
                    std::vector<uint8_t> data(size);
                    heif_image_handle_get_metadata(handle, metadata_ids[j], data.data());
                    
                    std::cout << "      First 16 bytes (hex): ";
                    for (size_t k = 0; k < std::min(size_t(16), size); k++) {
                        printf("%02x ", data[k]);
                    }
                    std::cout << std::endl;
                    
                    // Try to parse
                    auto meta = MultiLevelPaddingMetadata::deserialize(data.data(), size);
                    if (meta) {
                        std::cout << "      ✓ Valid mtpd metadata!" << std::endl;
                    } else {
                        std::cout << "      ✗ Failed to parse" << std::endl;
                    }
                }
            }
        }
        
        heif_image_handle_release(handle);
    }
    
    heif_context_free(ctx);
    std::cout << "\n" << std::string(60, '=') << "\n" << std::endl;
}
#endif // DEBUG_READ_ALL_METADATA_H

In [ ]:
//debug_all_metadata("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif");

## Usage Examples for encode_pyramid with order support

In [ ]:
#ifndef HEIF_PYRAMID_USAGE_EXAMPLES_FIXED_H
#define HEIF_PYRAMID_USAGE_EXAMPLES_FIXED_H

// Fixed usage examples - no memset needed, use constructor instead

// Example 1: Default order (memory-efficient, high-res first)
void example_pyramid_default(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_default.heif";
    
    PyramidConfig config;  // Constructor initializes everything
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::TILI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::DEFLATE;
    config.quality = 100;
    config.output_bit_depth = 8;
    config.padding_value = 0;
    config.write_order = PyramidWriteOrder::Default;  // Memory-efficient
    config.store_padding_metadata = true;
    config.padding_format = PaddingMetadataFormat::TPAD_BINARY;
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 2: COG-style order (coarse first)
void example_pyramid_cog(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_cog.heif";
    
    PyramidConfig config;  // Constructor initializes everything
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::TILI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::DEFLATE;
    config.quality = 100;
    config.output_bit_depth = 8;
    config.padding_value = 255;  // White padding
    config.write_order = PyramidWriteOrder::CoarseFirst;  // COG-style
    config.store_padding_metadata = true;
    config.padding_format = PaddingMetadataFormat::BOTH;  // Binary + XMP
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 3: Grid tiling with default order
void example_pyramid_grid(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_grid.heif";
    
    PyramidConfig config;
    config.num_levels = 4;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::Grid;
    config.tile_width = 512;
    config.tile_height = 512;
    config.compression = ExtendedCompressionType::HEVC;
    config.quality = 70;
    config.output_bit_depth = 8;
    config.write_order = PyramidWriteOrder::Default;
    config.store_padding_metadata = true;
    config.padding_format = PaddingMetadataFormat::TPAD_BINARY;
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 4: UNCI tiling with COG order
void example_pyramid_unci_cog(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_unci_cog.heif";
    
    PyramidConfig config;
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::UNCI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::DEFLATE;
    config.quality = 100;
    config.output_bit_depth = 8;
    config.padding_value = 128;  // Mid-gray
    config.write_order = PyramidWriteOrder::CoarseFirst;  // COG
    config.store_padding_metadata = true;
    config.padding_format = PaddingMetadataFormat::XMP_TEXT;
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 5: Non-tiled pyramid (no padding metadata needed)
void example_pyramid_nontiled(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_nontiled.heif";
    
    PyramidConfig config;
    config.num_levels = 4;
    config.use_tiling = false;  // No tiling
    config.compression = ExtendedCompressionType::AV1;
    config.quality = 60;
    config.output_bit_depth = 8;
    config.write_order = PyramidWriteOrder::CoarseFirst;
    config.store_padding_metadata = false;  // No padding, no metadata needed
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 6: Variable quality per level
void example_pyramid_variable_quality(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_varquality.heif";
    
    PyramidConfig config;
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::TILI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::HEVC;
    config.quality = 70;  // Default
    config.quality_per_level = {95, 85, 75, 65, 55};  // Higher quality for full-res
    config.output_bit_depth = 8;
    config.write_order = PyramidWriteOrder::Default;
    config.store_padding_metadata = true;
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 7: Custom offset table configuration with COG
void example_pyramid_custom_offset_cog(const char* input_file, const char* output_dir) {
    std::string output_path = std::string(output_dir) + "/pyramid_custom_offset_cog.heif";
    
    PyramidConfig config;
    config.num_levels = 5;
    config.use_tiling = true;
    config.tiling_scheme = TilingScheme::TILI;
    config.tile_width = 256;
    config.tile_height = 256;
    config.compression = ExtendedCompressionType::DEFLATE;
    config.quality = 100;
    config.output_bit_depth = 8;
    config.padding_value = 0;
    config.write_order = PyramidWriteOrder::CoarseFirst;  // COG-style
    
    // Custom offset table configuration
    config.offset_config = OffsetTableConfig(40, 32, false);  // 40-bit offset, 32-bit size
    
    config.store_padding_metadata = true;
    config.padding_format = PaddingMetadataFormat::BOTH;
    
    encode_pyramid_with_config(input_file, output_path.c_str(), config);
}

// Example 8: Comparison - same config, different write orders
void example_compare_write_orders(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Comparing Write Orders ===" << std::endl;
    
    // Base configuration
    PyramidConfig base_config;
    base_config.num_levels = 4;
    base_config.use_tiling = true;
    base_config.tiling_scheme = TilingScheme::TILI;
    base_config.tile_width = 256;
    base_config.tile_height = 256;
    base_config.compression = ExtendedCompressionType::DEFLATE;
    base_config.quality = 100;
    base_config.output_bit_depth = 8;
    base_config.padding_value = 0;
    base_config.store_padding_metadata = true;
    base_config.padding_format = PaddingMetadataFormat::TPAD_BINARY;
    
    // Test 1: Default order
    {
        PyramidConfig config = base_config;
        config.write_order = PyramidWriteOrder::Default;
        std::string output_path = std::string(output_dir) + "/compare_default.heif";
        
        std::cout << "\n--- Test 1: Default Order (Memory-Efficient) ---" << std::endl;
        encode_pyramid_with_config(input_file, output_path.c_str(), config);
    }
    
    // Test 2: COG order
    {
        PyramidConfig config = base_config;
        config.write_order = PyramidWriteOrder::CoarseFirst;
        std::string output_path = std::string(output_dir) + "/compare_cog.heif";
        
        std::cout << "\n--- Test 2: COG Order (Coarse First) ---" << std::endl;
        encode_pyramid_with_config(input_file, output_path.c_str(), config);
    }
    
    std::cout << "\n✓ Comparison complete. Check file sizes and structures." << std::endl;
}

// Run all examples
void run_all_pyramid_examples(const char* input_file, const char* output_dir) {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "Running All Pyramid Encoding Examples" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    
    try {
        example_pyramid_default(input_file, output_dir);
        example_pyramid_cog(input_file, output_dir);
        example_pyramid_grid(input_file, output_dir);
        example_pyramid_unci_cog(input_file, output_dir);
        example_pyramid_nontiled(input_file, output_dir);
        example_pyramid_variable_quality(input_file, output_dir);
        example_pyramid_custom_offset_cog(input_file, output_dir);
        example_compare_write_orders(input_file, output_dir);
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "✓ All pyramid examples completed successfully!" << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during examples: " << e.what() << std::endl;
    }
}

std::cout << "Pyramid encoding examples defined (fixed - no memset)." << std::endl;

#endif // HEIF_PYRAMID_USAGE_EXAMPLES_FIXED_H

## Summary

### Key Features:

1. **Protection against redefinition**: All code blocks wrapped with `#ifndef` guards

2. **Comprehensive compression support**:
   - Video codecs: HEVC, AV1, VVC, AVC
   - Image codecs: JPEG, JPEG2000, HTJ2K
   - Lossless: DEFLATE, ZLIB, Brotli, ZSTD, LZW, LERC, PackBits
   - GDAL integration for COG/GeoTIFF compression compatibility

3. **Detailed file structure analysis**:
   - Complete box hierarchy visualization
   - Offset and size information for each box
   - Box descriptions and purpose
   - Summary statistics showing space usage
   - Percentage breakdown of file components

4. **Flexible encoding options**:
   - Non-tiled images
   - Tiled images (grid layout)
   - Pyramid/multi-resolution images
   - All with configurable compression

5. **GDAL integration**:
   - Full GeoTIFF support
   - Metadata preservation
   - Projection information
   - Support for various data types

### Usage Notes:
- Cells can be re-executed safely without redefinition errors
- Each encoding automatically analyzes the output file structure
- Compression algorithms list shows what's supported by libheif and GDAL
- File analyzer provides detailed breakdown of mdat, meta, and all other boxes